# 10 – Single-Factor Benchmark Challenge

This notebook tests the two retained standalone factors against the three shortlisted portfolios under a controlled, common implementation framework. SPY remains a contextual long-only benchmark.

## Objective

The analysis asks whether the existing final hierarchy remains justified after adding **Momentum Only** and **Realised Volatility Only** as controlled component-factor benchmarks. Comparisons emphasise matched dates, frequencies, costs, rebalance phases, exposure budgets, and accounting conventions.

## Pre-registered expectations

Before inspecting the controlled results:

1. Composite Score is expected to remain preferable to Realised Volatility Only on combined return and risk evidence.
2. Realised Volatility Only may exceed the sleeve portfolios in raw return but is expected to carry greater risk or concentration.
3. Momentum Only is expected to remain weak as a standalone portfolio.
4. Momentum may still improve Composite Score through ranking interaction or diversification.
5. The existing hierarchy is expected to remain defensible, but contradictory evidence will be reported directly.

These are testable expectations, not target conclusions.

## Frozen research design

Factor definitions, directions, signal processing, quintiles, gross budgets, forward-return alignment, drift-aware holdings, full-L1 turnover, linear transaction costs, missing-return treatment, benchmark treatment, and numerical tolerance remain unchanged. The controlled window is **2016-01-07 to 2026-07-01**, and the baseline cost is **10 bps per unit of turnover**. No data are downloaded and no existing artifact is written.

## 1. Setup and input audits

The audit establishes file availability, schemas, unique keys, and the exact shared date index before any portfolio is reconstructed.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from IPython.display import display

from alpha_research.attribution import (
    prepare_security_attribution,
    reconcile_security_attribution,
)
from alpha_research.backtest import (
    BacktestConfig,
    run_long_short_backtest,
    run_target_weight_backtest,
    summarise_backtest,
    summarise_backtest_subperiods,
)
from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.config.research import (
    BACKTEST_RETURN_COLUMN,
    BASELINE_TRANSACTION_COST_BPS,
    DEFAULT_NUMERICAL_TOLERANCE,
    MONITORING_SPECIFICATION,
    ROBUSTNESS_REBALANCE_FREQUENCIES,
    ROBUSTNESS_TRANSACTION_COST_GRID_BPS,
    STRATEGY_EVALUATION_START_DATE,
    STRATEGY_SPECIFICATIONS,
    TRADING_DAYS_PER_YEAR,
)
from alpha_research.dashboard_analytics import (
    build_side_cost_attribution_summary,
    prepare_performance_history,
)
from alpha_research.data_loader import load_parquet
from alpha_research.metrics import summarise_returns
from alpha_research.monitoring import (
    calculate_implementation_monitoring_state,
    calculate_performance_risk_state,
)
from alpha_research.risk import (
    calculate_beta_state,
    calculate_concentration_state,
    calculate_sector_exposure,
    prepare_holdings_beta_detail,
    summarise_sector_exposure,
)
from alpha_research.visualisation import (
    PORTFOLIO_COLOURS,
    build_cumulative_performance_figure,
    build_drawdown_figure,
    build_rolling_metric_figure,
    build_side_cost_attribution_figure,
)
from alpha_research.workflows import (
    build_common_strategy_backtests,
    build_common_strategy_robustness_grid,
    build_frozen_strategy_target_weights,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

CHALLENGE_END_DATE = pd.Timestamp("2026-07-01")
AUDIT_TOLERANCE = DEFAULT_NUMERICAL_TOLERANCE
ACTIVE_PORTFOLIO_ORDER = (
    "Momentum Only",
    "Realised Volatility Only",
    "Composite Score",
    "Fixed 50/50 Sleeves",
    "Pure Inverse Volatility",
)
COMPARISON_ORDER = (*ACTIVE_PORTFOLIO_ORDER, "SPY")
LEGACY_NAME_MAP = {
    "12-1 Momentum": "Momentum Only",
    "Realised Volatility": "Realised Volatility Only",
}

In [2]:
ARTIFACT_PATHS = {
    "factor_panel": PROCESSED_DATA_DIR / "factor_panel.parquet",
    "factor_backtest_summary": PROCESSED_DATA_DIR / "factor_backtest_summary.parquet",
    "momentum_daily": PROCESSED_DATA_DIR / "backtest_12_1_momentum_daily.parquet",
    "momentum_holdings": PROCESSED_DATA_DIR / "backtest_12_1_momentum_holdings.parquet",
    "volatility_daily": PROCESSED_DATA_DIR / "backtest_realised_volatility_daily.parquet",
    "volatility_holdings": PROCESSED_DATA_DIR / "backtest_realised_volatility_holdings.parquet",
    "five_day_candidates": PROCESSED_DATA_DIR / "portfolio_optimisation_benchmarks.parquet",
    "selected_candidate_daily": PROCESSED_DATA_DIR / "attribution_portfolio_daily.parquet",
    "benchmark_daily": PROCESSED_DATA_DIR / "attribution_benchmark_daily.parquet",
}

missing_paths = [str(path) for path in ARTIFACT_PATHS.values() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Required local artifacts are missing: {missing_paths}")

artifact_manifest = pd.DataFrame(
    [
        {"dataset": name, "size_bytes": path.stat().st_size}
        for name, path in ARTIFACT_PATHS.items()
    ]
)

factor_panel = load_parquet(ARTIFACT_PATHS["factor_panel"])
stored_factor_summary = load_parquet(ARTIFACT_PATHS["factor_backtest_summary"])
stored_legacy_daily = {
    "12-1 Momentum": load_parquet(ARTIFACT_PATHS["momentum_daily"]),
    "Realised Volatility": load_parquet(ARTIFACT_PATHS["volatility_daily"]),
}
stored_legacy_holdings = {
    "12-1 Momentum": load_parquet(ARTIFACT_PATHS["momentum_holdings"]),
    "Realised Volatility": load_parquet(ARTIFACT_PATHS["volatility_holdings"]),
}
five_day_candidate_artifact = load_parquet(ARTIFACT_PATHS["five_day_candidates"])
selected_candidate_daily = load_parquet(ARTIFACT_PATHS["selected_candidate_daily"])
benchmark_daily = load_parquet(ARTIFACT_PATHS["benchmark_daily"])

display(artifact_manifest)

,dataset,size_bytes
0,factor_panel,79031867
1,factor_backtest_summary,8469
2,momentum_daily,184309
3,momentum_holdings,92164
4,volatility_daily,195253
5,volatility_holdings,89457
6,five_day_candidates,520285
7,selected_candidate_daily,798207
8,benchmark_daily,47255


In [3]:
REQUIRED_FACTOR_COLUMNS = {
    "date",
    "ticker",
    BACKTEST_RETURN_COLUMN,
    "mom_12_1m_z",
    "realised_vol_63_z",
    "sector",
    "beta_126",
    "dollar_volume",
}
missing_factor_columns = REQUIRED_FACTOR_COLUMNS - set(factor_panel.columns)
if missing_factor_columns:
    raise KeyError(f"factor_panel is missing columns: {sorted(missing_factor_columns)}")

factor_panel["date"] = pd.to_datetime(factor_panel["date"], errors="raise")
five_day_candidate_artifact["date"] = pd.to_datetime(
    five_day_candidate_artifact["date"], errors="raise"
)
selected_candidate_daily["date"] = pd.to_datetime(
    selected_candidate_daily["date"], errors="raise"
)
benchmark_daily["date"] = pd.to_datetime(benchmark_daily["date"], errors="raise")

if factor_panel.duplicated(["date", "ticker"]).any():
    raise ValueError("factor_panel contains duplicate date-ticker keys.")
if five_day_candidate_artifact.duplicated(["portfolio", "date"]).any():
    raise ValueError("five_day_candidate_artifact contains duplicate keys.")
if selected_candidate_daily.duplicated(["portfolio", "date"]).any():
    raise ValueError("selected_candidate_daily contains duplicate keys.")
if benchmark_daily.duplicated(["benchmark", "date"]).any():
    raise ValueError("benchmark_daily contains duplicate keys.")
if benchmark_daily["benchmark"].drop_duplicates().tolist() != ["SPY"]:
    raise ValueError("Expected one SPY benchmark series.")

common_dates = pd.DatetimeIndex(benchmark_daily["date"]).sort_values()
if common_dates.min() != STRATEGY_EVALUATION_START_DATE:
    raise ValueError("Unexpected common-window start date.")
if common_dates.max() != CHALLENGE_END_DATE:
    raise ValueError("Unexpected common-window end date.")

required_candidate_names = set(ACTIVE_PORTFOLIO_ORDER[2:])
for source_name, source in {
    "five_day_candidates": five_day_candidate_artifact.loc[
        five_day_candidate_artifact["portfolio"].isin(required_candidate_names)
    ],
    "selected_candidates": selected_candidate_daily,
}.items():
    for portfolio, portfolio_data in source.groupby("portfolio", sort=False):
        dates = pd.DatetimeIndex(portfolio_data["date"]).sort_values()
        if not dates.equals(common_dates):
            raise ValueError(f"{source_name}: {portfolio} dates do not match the common window.")

input_audit_rows = [
    {
        "dataset": "factor_panel",
        "rows": len(factor_panel),
        "start_date": factor_panel["date"].min(),
        "end_date": factor_panel["date"].max(),
        "duplicate_keys": int(factor_panel.duplicated(["date", "ticker"]).sum()),
    },
    {
        "dataset": "benchmark_daily",
        "rows": len(benchmark_daily),
        "start_date": benchmark_daily["date"].min(),
        "end_date": benchmark_daily["date"].max(),
        "duplicate_keys": int(benchmark_daily.duplicated(["benchmark", "date"]).sum()),
    },
]
for factor_name, daily in stored_legacy_daily.items():
    daily["date"] = pd.to_datetime(daily["date"], errors="raise")
    input_audit_rows.append(
        {
            "dataset": f"legacy_daily: {factor_name}",
            "rows": len(daily),
            "start_date": daily["date"].min(),
            "end_date": daily["date"].max(),
            "duplicate_keys": int(daily["date"].duplicated().sum()),
        }
    )

input_audit = pd.DataFrame(input_audit_rows)
display(input_audit)
print(f"Common evaluation dates: {len(common_dates):,}")

,dataset,rows,start_date,end_date,duplicate_keys
0,factor_panel,284249,2015-01-02,2026-07-02,0
1,benchmark_daily,2635,2016-01-07,2026-07-01,0
2,legacy_daily: 12-1 Momentum,2890,2015-01-02,2026-07-01,0
3,legacy_daily: Realised Volatility,2890,2015-01-02,2026-07-01,0


Common evaluation dates: 2,635


### Setup findings

All required local inputs are present. Security and portfolio keys are unique, and the final SPY series defines 2,635 common trading dates from 2016-01-07 through 2026-07-01. The factor panel extends one row-date further because the last price date supplies the forward return realised after 2026-07-01.

## 2. Baseline reproduction

The first audit replays the original five-day standalone script on its full historical window. The second replays the three frozen selected implementations on the final common window.

In [4]:
legacy_factor_columns = {
    "12-1 Momentum": "mom_12_1m_z",
    "Realised Volatility": "realised_vol_63_z",
}
replayed_legacy_daily = {}
replayed_legacy_holdings = {}
replayed_summary_rows = []
for factor_name, factor_column in legacy_factor_columns.items():
    daily, holdings = run_long_short_backtest(
        factor_panel,
        factor_column=factor_column,
        config=BacktestConfig(),
    )
    replayed_legacy_daily[factor_name] = daily
    replayed_legacy_holdings[factor_name] = holdings
    summary = summarise_backtest(daily).iloc[0].to_dict()
    summary["factor"] = factor_name
    replayed_summary_rows.append(summary)
replayed_factor_summary = pd.DataFrame(replayed_summary_rows).set_index("factor")
stored_summary_indexed = stored_factor_summary.set_index("factor").sort_index()
replayed_summary_indexed = replayed_factor_summary.sort_index()
legacy_audit_rows = []

for factor_name in LEGACY_NAME_MAP:
    replayed_daily = replayed_legacy_daily[factor_name].sort_values("date").reset_index(drop=True)
    stored_daily = stored_legacy_daily[factor_name].sort_values("date").reset_index(drop=True)
    replayed_holdings = replayed_legacy_holdings[factor_name].sort_values(
        ["date", "ticker"]
    ).reset_index(drop=True)
    stored_holdings = stored_legacy_holdings[factor_name].sort_values(
        ["date", "ticker"]
    ).reset_index(drop=True)

    daily_numeric_columns = [
        column for column in stored_daily.columns if column not in {"date", "is_rebalance"}
    ]
    summary_columns = list(stored_summary_indexed.select_dtypes(include="number").columns)
    daily_difference = max(
        (replayed_daily[column] - stored_daily[column]).abs().max()
        for column in daily_numeric_columns
    )
    holdings_difference = (replayed_holdings["weight"] - stored_holdings["weight"]).abs().max()
    summary_difference = max(
        abs(
            replayed_summary_indexed.loc[factor_name, column]
            - stored_summary_indexed.loc[factor_name, column]
        )
        for column in summary_columns
    )
    keys_match = replayed_holdings[["date", "ticker"]].equals(
        stored_holdings[["date", "ticker"]]
    )
    dates_match = replayed_daily["date"].equals(stored_daily["date"])
    flags_match = replayed_daily["is_rebalance"].equals(stored_daily["is_rebalance"])

    legacy_audit_rows.append(
        {
            "portfolio": LEGACY_NAME_MAP[factor_name],
            "observations": len(replayed_daily),
            "start_date": replayed_daily["date"].min(),
            "end_date": replayed_daily["date"].max(),
            "maximum_daily_difference": daily_difference,
            "maximum_holdings_difference": holdings_difference,
            "maximum_summary_difference": summary_difference,
            "audit_passes": bool(
                keys_match
                and dates_match
                and flags_match
                and max(daily_difference, holdings_difference, summary_difference)
                < AUDIT_TOLERANCE
            ),
        }
    )

legacy_reproduction_audit = pd.DataFrame(legacy_audit_rows)
if not legacy_reproduction_audit["audit_passes"].all():
    raise ValueError("The original standalone backtests did not reproduce.")

original_standalone_headlines = (
    stored_factor_summary.assign(
        portfolio=lambda data: data["factor"].map(LEGACY_NAME_MAP)
    )
    .set_index("portfolio")
    .reindex(ACTIVE_PORTFOLIO_ORDER[:2])
    .reset_index()[
        [
            "portfolio",
            "observations",
            "annualised_return",
            "annualised_volatility",
            "sharpe_ratio",
            "max_drawdown",
        ]
    ]
)

display(legacy_reproduction_audit)
display(original_standalone_headlines.round(6))

,portfolio,observations,start_date,end_date,maximum_daily_difference,maximum_holdings_difference,maximum_summary_difference,audit_passes
0,Momentum Only,2890,2015-01-02,2026-07-01,0.0,0.0,0.0,True
1,Realised Volatility Only,2890,2015-01-02,2026-07-01,0.0,0.0,0.0,True


,portfolio,observations,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown
0,Momentum Only,2890.0,0.018785,0.211280,0.194719,-0.490626
1,Realised Volatility Only,2890.0,0.157869,0.239941,0.731003,-0.452815


In [5]:
frozen_targets = build_frozen_strategy_target_weights(factor_panel)
strategy_lookup = {item.portfolio: item for item in STRATEGY_SPECIFICATIONS}
candidate_reproduction_rows = []
replayed_frozen_daily = {}
CANDIDATE_AUDIT_COLUMNS = [
    "long_return",
    "short_return",
    "gross_return",
    "turnover",
    "transaction_cost",
    "net_return",
    "long_exposure",
    "short_exposure",
    "net_exposure",
    "gross_exposure",
    "missing_return_weight",
    "gross_cumulative_return",
    "net_cumulative_return",
]
return_panel = factor_panel[["date", "ticker", BACKTEST_RETURN_COLUMN]].copy()

for portfolio, targets in frozen_targets.items():
    specification = strategy_lookup[portfolio]
    replayed, _ = run_target_weight_backtest(
        return_panel,
        targets,
        transaction_cost_bps=specification.transaction_cost_bps,
    )
    replayed = replayed.loc[replayed["date"].isin(common_dates)].sort_values("date").reset_index(drop=True)
    replayed["gross_cumulative_return"] = (1.0 + replayed["gross_return"]).cumprod()
    replayed["net_cumulative_return"] = (1.0 + replayed["net_return"]).cumprod()
    reference = selected_candidate_daily.loc[
        selected_candidate_daily["portfolio"].eq(portfolio)
    ].sort_values("date").reset_index(drop=True)
    maximum_difference = max(
        (replayed[column] - reference[column]).abs().max()
        for column in CANDIDATE_AUDIT_COLUMNS
    )
    dates_match = replayed["date"].equals(reference["date"])
    flags_match = replayed["is_rebalance"].equals(reference["is_rebalance"])
    candidate_reproduction_rows.append(
        {
            "portfolio": portfolio,
            "rebalance_frequency": specification.rebalance_frequency,
            "rebalance_offset": specification.rebalance_offset,
            "observations": len(replayed),
            "maximum_absolute_difference": maximum_difference,
            "audit_passes": bool(
                dates_match and flags_match and maximum_difference < AUDIT_TOLERANCE
            ),
        }
    )
    replayed_frozen_daily[portfolio] = replayed

candidate_reproduction_audit = pd.DataFrame(candidate_reproduction_rows)
if not candidate_reproduction_audit["audit_passes"].all():
    raise ValueError("The frozen selected candidates did not reproduce.")

frozen_candidate_headlines = []
for portfolio in ACTIVE_PORTFOLIO_ORDER[2:]:
    summary = summarise_backtest(replayed_frozen_daily[portfolio]).iloc[0]
    frozen_candidate_headlines.append(
        {
            "portfolio": portfolio,
            "annualised_return": summary["annualised_return"],
            "annualised_volatility": summary["annualised_volatility"],
            "sharpe_ratio": summary["sharpe_ratio"],
            "max_drawdown": summary["max_drawdown"],
        }
    )

display(candidate_reproduction_audit)
display(pd.DataFrame(frozen_candidate_headlines).round(6))

,portfolio,rebalance_frequency,rebalance_offset,observations,maximum_absolute_difference,audit_passes
0,Composite Score,21,0,2635,0.0,True
1,Fixed 50/50 Sleeves,10,0,2635,0.0,True
2,Pure Inverse Volatility,10,0,2635,0.0,True


,portfolio,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown
0,Composite Score,0.161268,0.211671,0.812777,-0.292379
1,Fixed 50/50 Sleeves,0.108436,0.167656,0.698280,-0.256835
2,Pure Inverse Volatility,0.109593,0.162527,0.721505,-0.217693


### Reproduction findings

Both original standalone artifacts reproduce on their longer sample, and all three frozen candidates reproduce on the final common window within the project tolerance. The original standalone engine holds weights constant between rebalance dates; the controlled challenge below instead uses the final drift-aware target-weight engine for all five active portfolios. Restricting the legacy returns to the common dates would therefore not be an identical-framework test.

## 3. Common-window comparison

All five active portfolios are now reconstructed at a five-day frequency, offset zero, and 10 bps. SPY is included without an active-strategy cost model and should be interpreted as long-only market context.

In [6]:
COMMON_BASELINE_CONFIG = BacktestConfig(
    rebalance_frequency=5,
    rebalance_offset=0,
    transaction_cost_bps=BASELINE_TRANSACTION_COST_BPS,
)
common_daily_full, common_holdings_full, common_targets_full = (
    build_common_strategy_backtests(factor_panel, COMMON_BASELINE_CONFIG)
)

common_daily = {}
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_daily_full[portfolio].loc[
        common_daily_full[portfolio]["date"].isin(common_dates)
    ].sort_values("date").reset_index(drop=True)
    daily["gross_cumulative_return"] = (1.0 + daily["gross_return"]).cumprod()
    daily["net_cumulative_return"] = (1.0 + daily["net_return"]).cumprod()
    common_daily[portfolio] = daily

common_baseline_daily = pd.concat(
    [
        common_daily[portfolio].assign(
            portfolio=portfolio,
            rebalance_frequency=COMMON_BASELINE_CONFIG.rebalance_frequency,
            rebalance_offset=COMMON_BASELINE_CONFIG.rebalance_offset,
            transaction_cost_bps=COMMON_BASELINE_CONFIG.transaction_cost_bps,
        )
        for portfolio in ACTIVE_PORTFOLIO_ORDER
    ],
    ignore_index=True,
).sort_values(["portfolio", "date"]).reset_index(drop=True)

print(f"Controlled baseline rows: {len(common_baseline_daily):,}")

Controlled baseline rows: 13,175


In [7]:
target_audit_rows = []
accounting_audit_rows = []
five_day_reproduction_rows = []

for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_daily[portfolio]
    holdings = common_holdings_full[portfolio].loc[
        common_holdings_full[portfolio]["date"].isin(common_dates)
    ].copy()
    targets = common_targets_full[portfolio].loc[
        common_targets_full[portfolio]["date"].isin(common_dates)
    ].copy()

    target_state = targets.groupby("date")["weight"].agg(
        long_gross=lambda weights: weights.clip(lower=0.0).sum(),
        short_gross=lambda weights: -weights.clip(upper=0.0).sum(),
        net_exposure="sum",
        gross_exposure=lambda weights: weights.abs().sum(),
    )
    exact_budget_required = portfolio in ACTIVE_PORTFOLIO_ORDER[:3]
    exact_budget_passes = bool(
        not exact_budget_required
        or (
            np.allclose(target_state["long_gross"], 1.0, atol=AUDIT_TOLERANCE)
            and np.allclose(target_state["short_gross"], 1.0, atol=AUDIT_TOLERANCE)
        )
    )
    target_audit_rows.append(
        {
            "portfolio": portfolio,
            "rebalance_dates": targets["date"].nunique(),
            "maximum_absolute_net_target": target_state["net_exposure"].abs().max(),
            "minimum_target_gross": target_state["gross_exposure"].min(),
            "maximum_target_gross": target_state["gross_exposure"].max(),
            "audit_passes": bool(
                not targets.duplicated(["date", "ticker"]).any()
                and target_state["net_exposure"].abs().max() < AUDIT_TOLERANCE
                and target_state["gross_exposure"].le(2.0 + AUDIT_TOLERANCE).all()
                and exact_budget_passes
            ),
        }
    )

    holdings_state = (
        holdings.assign(
            long_weight=lambda data: data["weight"].clip(lower=0.0),
            short_weight=lambda data: -data["weight"].clip(upper=0.0),
            absolute_trade=lambda data: data["trade"].abs(),
        )
        .groupby("date")
        .agg(
            holdings_long_exposure=("long_weight", "sum"),
            holdings_short_exposure=("short_weight", "sum"),
            holdings_turnover=("absolute_trade", "sum"),
        )
        .reset_index()
    )
    reconciliation = daily.merge(holdings_state, on="date", validate="one_to_one")
    maximum_accounting_difference = max(
        (daily["long_return"] + daily["short_return"] - daily["gross_return"]).abs().max(),
        (daily["gross_return"] - daily["transaction_cost"] - daily["net_return"]).abs().max(),
        (
            daily["turnover"] * COMMON_BASELINE_CONFIG.transaction_cost_bps / 10_000.0
            - daily["transaction_cost"]
        ).abs().max(),
        (reconciliation["long_exposure"] - reconciliation["holdings_long_exposure"]).abs().max(),
        (reconciliation["short_exposure"] - reconciliation["holdings_short_exposure"]).abs().max(),
        (reconciliation["turnover"] - reconciliation["holdings_turnover"]).abs().max(),
    )
    accounting_audit_rows.append(
        {
            "portfolio": portfolio,
            "observations": len(daily),
            "maximum_absolute_difference": maximum_accounting_difference,
            "maximum_missing_return_weight": daily["missing_return_weight"].max(),
            "audit_passes": bool(
                pd.DatetimeIndex(daily["date"]).equals(common_dates)
                and not holdings.duplicated(["date", "ticker"]).any()
                and maximum_accounting_difference < AUDIT_TOLERANCE
                and daily["missing_return_weight"].max() < AUDIT_TOLERANCE
            ),
        }
    )

    if portfolio in required_candidate_names:
        reference = five_day_candidate_artifact.loc[
            five_day_candidate_artifact["portfolio"].eq(portfolio)
        ].sort_values("date").reset_index(drop=True)
        audit_columns = [
            "gross_return",
            "net_return",
            "turnover",
            "transaction_cost",
            "missing_return_weight",
            "gross_exposure",
            "net_exposure",
        ]
        maximum_difference = max(
            (daily[column] - reference[column]).abs().max() for column in audit_columns
        )
        five_day_reproduction_rows.append(
            {
                "portfolio": portfolio,
                "maximum_absolute_difference": maximum_difference,
                "rebalance_flags_match": daily["is_rebalance"].equals(reference["is_rebalance"]),
                "audit_passes": bool(
                    maximum_difference < AUDIT_TOLERANCE
                    and daily["is_rebalance"].equals(reference["is_rebalance"])
                ),
            }
        )

target_weight_audit = pd.DataFrame(target_audit_rows)
accounting_audit = pd.DataFrame(accounting_audit_rows)
five_day_candidate_reproduction_audit = pd.DataFrame(five_day_reproduction_rows)

if not target_weight_audit["audit_passes"].all():
    raise ValueError("Target-weight constraints failed.")
if not accounting_audit["audit_passes"].all():
    raise ValueError("Portfolio accounting failed.")
if not five_day_candidate_reproduction_audit["audit_passes"].all():
    raise ValueError("The controlled candidates do not reproduce the five-day artifact.")

display(target_weight_audit)
display(accounting_audit)
display(five_day_candidate_reproduction_audit)

,portfolio,rebalance_dates,maximum_absolute_net_target,minimum_target_gross,maximum_target_gross,audit_passes
0,Momentum Only,527,0.000000e+00,2.000000,2.0,True
1,Realised Volatility Only,527,0.000000e+00,2.000000,2.0,True
2,Composite Score,527,0.000000e+00,2.000000,2.0,True
3,Fixed 50/50 Sleeves,527,0.000000e+00,0.450000,2.0,True
4,Pure Inverse Volatility,527,4.857226e-17,0.485807,2.0,True


,portfolio,observations,maximum_absolute_difference,maximum_missing_return_weight,audit_passes
0,Momentum Only,2635,2.220446e-16,0.0,True
1,Realised Volatility Only,2635,2.220446e-16,0.0,True
2,Composite Score,2635,2.220446e-16,0.0,True
3,Fixed 50/50 Sleeves,2635,2.220446e-16,0.0,True
4,Pure Inverse Volatility,2635,2.220446e-16,0.0,True


,portfolio,maximum_absolute_difference,rebalance_flags_match,audit_passes
0,Composite Score,0.000000e+00,True,True
1,Fixed 50/50 Sleeves,0.000000e+00,True,True
2,Pure Inverse Volatility,1.110223e-15,True,True


In [8]:
baseline_rows = []
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_daily[portfolio]
    net_summary = summarise_backtest(daily, return_column="net_return").iloc[0]
    gross_summary = summarise_backtest(daily, return_column="gross_return").iloc[0]
    baseline_rows.append(
        {
            "portfolio": portfolio,
            "observations": int(net_summary["observations"]),
            "total_return": net_summary["total_return"],
            "annualised_return": net_summary["annualised_return"],
            "annualised_volatility": net_summary["annualised_volatility"],
            "sharpe_ratio": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "positive_day_fraction": net_summary["positive_day_fraction"],
            "average_daily_turnover": net_summary["average_daily_turnover"],
            "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
            "cumulative_transaction_cost": net_summary["total_transaction_cost"],
            "annualised_return_cost_drag": (
                gross_summary["annualised_return"] - net_summary["annualised_return"]
            ),
            "average_gross_exposure": daily["gross_exposure"].mean(),
            "average_net_exposure": daily["net_exposure"].mean(),
        }
    )

spy_summary = summarise_returns(benchmark_daily["benchmark_return"])
baseline_rows.append(
    {
        "portfolio": "SPY",
        "observations": int(spy_summary["observations"]),
        "total_return": spy_summary["total_return"],
        "annualised_return": spy_summary["annualised_return"],
        "annualised_volatility": spy_summary["annualised_volatility"],
        "sharpe_ratio": spy_summary["sharpe_ratio"],
        "max_drawdown": spy_summary["maximum_drawdown"],
        "positive_day_fraction": spy_summary["positive_day_fraction"],
        "average_daily_turnover": 0.0,
        "average_rebalance_turnover": np.nan,
        "cumulative_transaction_cost": 0.0,
        "annualised_return_cost_drag": 0.0,
        "average_gross_exposure": 1.0,
        "average_net_exposure": 1.0,
    }
)

common_window_baseline = (
    pd.DataFrame(baseline_rows)
    .set_index("portfolio")
    .reindex(COMPARISON_ORDER)
    .reset_index()
)

display(common_window_baseline.round(6))

,portfolio,observations,total_return,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown,positive_day_fraction,average_daily_turnover,average_rebalance_turnover,cumulative_transaction_cost,annualised_return_cost_drag,average_gross_exposure,average_net_exposure
0,Momentum Only,2635,0.103780,0.009488,0.221888,0.154481,-0.509550,0.537381,0.111424,0.557119,0.293602,0.028744,2.002969,0.000579
1,Realised Volatility Only,2635,3.756356,0.160838,0.248639,0.724266,-0.458528,0.530930,0.082102,0.410509,0.216338,0.024269,2.000736,0.001164
2,Composite Score,2635,2.841731,0.137370,0.213330,0.710582,-0.306277,0.552182,0.104652,0.523261,0.275758,0.030386,2.000719,0.001304
3,Fixed 50/50 Sleeves,2635,1.801410,0.103533,0.168423,0.669545,-0.256562,0.544972,0.087102,0.435508,0.229513,0.024487,1.541543,0.001087
4,Pure Inverse Volatility,2635,1.816898,0.104115,0.162893,0.689863,-0.195652,0.551044,0.092385,0.461923,0.243433,0.026005,1.607862,0.001077
5,SPY,2635,3.533806,0.155530,0.178295,0.900386,-0.337173,0.554459,0.000000,NaN,0.000000,0.000000,1.000000,1.000000


In [9]:
performance_risk_state = calculate_performance_risk_state(
    common_baseline_daily[["date", "portfolio", "net_return"]],
    benchmark_daily[["date", "benchmark_return"]],
    portfolios=ACTIVE_PORTFOLIO_ORDER,
)
performance_history = prepare_performance_history(
    performance_risk_state,
    portfolios=COMPARISON_ORDER,
)

display(
    build_cumulative_performance_figure(
        performance_history,
        title="Common-Window Cumulative Performance – Five-Day Rebalancing",
    )
)
display(
    build_drawdown_figure(
        performance_history,
        title="Common-Window Drawdown – Five-Day Rebalancing",
    )
)

### Common-window findings

- **Realised Volatility leads the controlled strategy set:** 16.08% annualised return and 0.724 Sharpe, versus 0.95% and 0.154 for Momentum.
- **The blends trade return for risk control:** Composite reaches 13.74% annualised return with a 0.711 Sharpe; Fixed 50/50 and Pure Inverse Volatility return about 10.4%, with lower volatility and shallower drawdowns.
- **Pure Inverse Volatility has the shallowest strategy drawdown:** -19.57%, compared with -25.66% for Fixed 50/50, -30.63% for Composite, and -45.85% for Realised Volatility.
- **Trading costs remain material:** the 10 bps assumption reduces annualised returns by 2.43-3.04 percentage points across the five strategies.
- **SPY is a reference, not a like-for-like strategy:** it records 15.55% annualised return, a 0.900 Sharpe, and a -33.72% maximum drawdown without simulated transaction costs.

These results establish the controlled baseline. Frequency, cost, and market-phase robustness are evaluated next before drawing a final conclusion.

## 4. Frequency and cost sensitivity

The pre-specified grid covers 1, 5, 10, and 21 trading days; 0, 5, 10, 20, and 50 bps; and every valid offset. Each frequency-offset schedule is run once at zero cost, then recosted exactly from its unchanged turnover. This avoids post-hoc frequency selection and redundant portfolio reconstruction.

In [10]:
robustness_grid = build_common_strategy_robustness_grid(
    factor_panel,
    common_dates,
    rebalance_frequencies=ROBUSTNESS_REBALANCE_FREQUENCIES,
    transaction_cost_grid_bps=ROBUSTNESS_TRANSACTION_COST_GRID_BPS,
)

grid_key_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
    "transaction_cost_bps",
]
expected_grid_rows = (
    len(ACTIVE_PORTFOLIO_ORDER)
    * sum(ROBUSTNESS_REBALANCE_FREQUENCIES)
    * len(ROBUSTNESS_TRANSACTION_COST_GRID_BPS)
)
grid_construction_audit = (
    robustness_grid.groupby(
        ["portfolio", "rebalance_frequency"],
        sort=False,
    )
    .agg(
        offset_count=("rebalance_offset", "nunique"),
        cost_count=("transaction_cost_bps", "nunique"),
        variant_count=("transaction_cost_bps", "size"),
        minimum_observations=("observations", "min"),
        maximum_observations=("observations", "max"),
        start_date=("start_date", "min"),
        end_date=("end_date", "max"),
        maximum_missing_return_weight=("maximum_missing_return_weight", "max"),
        maximum_accounting_difference=("maximum_accounting_difference", "max"),
    )
    .reset_index()
)
grid_construction_audit["audit_passes"] = (
    grid_construction_audit["offset_count"].eq(
        grid_construction_audit["rebalance_frequency"]
    )
    & grid_construction_audit["cost_count"].eq(
        len(ROBUSTNESS_TRANSACTION_COST_GRID_BPS)
    )
    & grid_construction_audit["variant_count"].eq(
        grid_construction_audit["rebalance_frequency"]
        * len(ROBUSTNESS_TRANSACTION_COST_GRID_BPS)
    )
    & grid_construction_audit["minimum_observations"].eq(len(common_dates))
    & grid_construction_audit["maximum_observations"].eq(len(common_dates))
    & grid_construction_audit["start_date"].eq(common_dates.min())
    & grid_construction_audit["end_date"].eq(common_dates.max())
    & grid_construction_audit["maximum_missing_return_weight"].lt(AUDIT_TOLERANCE)
    & grid_construction_audit["maximum_accounting_difference"].lt(AUDIT_TOLERANCE)
)

if len(robustness_grid) != expected_grid_rows:
    raise ValueError("The robustness grid has an unexpected row count.")
if robustness_grid.duplicated(grid_key_columns).any():
    raise ValueError("The robustness grid contains duplicate variants.")
if not grid_construction_audit["audit_passes"].all():
    raise ValueError("The robustness construction audit failed.")

print(f"Validated robustness variants: {len(robustness_grid):,}")
display(grid_construction_audit)

Validated robustness variants: 925


,portfolio,rebalance_frequency,offset_count,cost_count,variant_count,minimum_observations,maximum_observations,start_date,end_date,maximum_missing_return_weight,maximum_accounting_difference,audit_passes
0,Momentum Only,1,1,5,5,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
1,Momentum Only,5,5,5,25,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
2,Momentum Only,10,10,5,50,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
3,Momentum Only,21,21,5,105,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
4,Realised Volatility Only,1,1,5,5,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
5,Realised Volatility Only,5,5,5,25,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
6,Realised Volatility Only,10,10,5,50,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
7,Realised Volatility Only,21,21,5,105,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
8,Composite Score,1,1,5,5,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True
9,Composite Score,5,5,5,25,2635,2635,2016-01-07,2026-07-01,0.0,0.0,True


In [11]:
offset_zero_frequency_cost = robustness_grid.loc[
    robustness_grid["rebalance_offset"].eq(0)
].copy()

cost_grid_audit = (
    offset_zero_frequency_cost.groupby(
        ["portfolio", "rebalance_frequency"],
        sort=False,
    )
    .agg(
        cost_count=("transaction_cost_bps", "nunique"),
        gross_return_range=(
            "gross_annualised_return",
            lambda values: values.max() - values.min(),
        ),
        turnover_range=(
            "average_daily_turnover",
            lambda values: values.max() - values.min(),
        ),
        gross_exposure_range=(
            "average_gross_exposure",
            lambda values: values.max() - values.min(),
        ),
        net_return_nonincreasing=(
            "net_annualised_return",
            lambda values: values.is_monotonic_decreasing,
        ),
        cost_drag_nondecreasing=(
            "annualised_return_cost_drag",
            lambda values: values.is_monotonic_increasing,
        ),
    )
    .reset_index()
)
cost_grid_audit["audit_passes"] = (
    cost_grid_audit["cost_count"].eq(len(ROBUSTNESS_TRANSACTION_COST_GRID_BPS))
    & cost_grid_audit["gross_return_range"].lt(AUDIT_TOLERANCE)
    & cost_grid_audit["turnover_range"].lt(AUDIT_TOLERANCE)
    & cost_grid_audit["gross_exposure_range"].lt(AUDIT_TOLERANCE)
    & cost_grid_audit["net_return_nonincreasing"]
    & cost_grid_audit["cost_drag_nondecreasing"]
)
if not cost_grid_audit["audit_passes"].all():
    raise ValueError("The transaction-cost grid audit failed.")

frequency_cost_columns = [
    "portfolio",
    "transaction_cost_bps",
    "gross_annualised_return",
    "net_annualised_return",
    "annualised_return_cost_drag",
    "net_annualised_volatility",
    "net_sharpe",
    "net_max_drawdown",
    "average_daily_turnover",
    "average_rebalance_turnover",
    "average_gross_exposure",
]

display(cost_grid_audit)
for frequency in ROBUSTNESS_REBALANCE_FREQUENCIES:
    print(f"Offset-zero frequency/cost results: {frequency} trading day(s)")
    display(
        offset_zero_frequency_cost.loc[
            offset_zero_frequency_cost["rebalance_frequency"].eq(frequency),
            frequency_cost_columns,
        ].round(6)
    )

,portfolio,rebalance_frequency,cost_count,gross_return_range,turnover_range,gross_exposure_range,net_return_nonincreasing,cost_drag_nondecreasing,audit_passes
0,Momentum Only,1,5,0.0,0.0,0.0,True,True,True
1,Momentum Only,5,5,0.0,0.0,0.0,True,True,True
2,Momentum Only,10,5,0.0,0.0,0.0,True,True,True
3,Momentum Only,21,5,0.0,0.0,0.0,True,True,True
4,Realised Volatility Only,1,5,0.0,0.0,0.0,True,True,True
5,Realised Volatility Only,5,5,0.0,0.0,0.0,True,True,True
6,Realised Volatility Only,10,5,0.0,0.0,0.0,True,True,True
7,Realised Volatility Only,21,5,0.0,0.0,0.0,True,True,True
8,Composite Score,1,5,0.0,0.0,0.0,True,True,True
9,Composite Score,5,5,0.0,0.0,0.0,True,True,True


Offset-zero frequency/cost results: 1 trading day(s)


,portfolio,transaction_cost_bps,gross_annualised_return,net_annualised_return,annualised_return_cost_drag,net_annualised_volatility,net_sharpe,net_max_drawdown,average_daily_turnover,average_rebalance_turnover,average_gross_exposure
0,Momentum Only,0.0,0.051605,0.051605,0.000000,0.221651,0.338752,-0.428050,0.253982,0.253982,2.000000
1,Momentum Only,5.0,0.051605,0.018481,0.033125,0.221674,0.194353,-0.484030,0.253982,0.253982,2.000000
2,Momentum Only,10.0,0.051605,-0.013606,0.065211,0.221701,0.049984,-0.559783,0.253982,0.253982,2.000000
3,Momentum Only,20.0,0.051605,-0.074791,0.126396,0.221767,-0.238637,-0.720489,0.253982,0.253982,2.000000
4,Momentum Only,50.0,0.051605,-0.236594,0.288199,0.222066,-1.102971,-0.957239,0.253982,0.253982,2.000000
185,Realised Volatility Only,0.0,0.188880,0.188880,0.000000,0.250238,0.816691,-0.428229,0.159444,0.159444,2.000000
186,Realised Volatility Only,5.0,0.188880,0.165248,0.023632,0.250228,0.736438,-0.438432,0.159444,0.159444,2.000000
187,Realised Volatility Only,10.0,0.188880,0.142083,0.046797,0.250222,0.656169,-0.448455,0.159444,0.159444,2.000000
188,Realised Volatility Only,20.0,0.188880,0.097117,0.091763,0.250220,0.495596,-0.467968,0.159444,0.159444,2.000000
189,Realised Volatility Only,50.0,0.188880,-0.027493,0.216374,0.250299,0.013858,-0.652042,0.159444,0.159444,2.000000


Offset-zero frequency/cost results: 5 trading day(s)


,portfolio,transaction_cost_bps,gross_annualised_return,net_annualised_return,annualised_return_cost_drag,net_annualised_volatility,net_sharpe,net_max_drawdown,average_daily_turnover,average_rebalance_turnover,average_gross_exposure
5,Momentum Only,0.0,0.038232,0.038232,0.000000,0.221899,0.281013,-0.475910,0.111424,0.557119,2.002969
6,Momentum Only,5.0,0.038232,0.023761,0.014471,0.221885,0.217756,-0.493007,0.111424,0.557119,2.002969
7,Momentum Only,10.0,0.038232,0.009488,0.028744,0.221888,0.154481,-0.509550,0.111424,0.557119,2.002969
8,Momentum Only,20.0,0.038232,-0.018478,0.056710,0.221943,0.027930,-0.541050,0.111424,0.557119,2.002969
9,Momentum Only,50.0,0.038232,-0.097907,0.136139,0.222500,-0.350731,-0.767310,0.111424,0.557119,2.002969
190,Realised Volatility Only,0.0,0.185107,0.185107,0.000000,0.248591,0.807636,-0.448071,0.082102,0.410509,2.000736
191,Realised Volatility Only,5.0,0.185107,0.172911,0.012196,0.248611,0.765960,-0.453324,0.082102,0.410509,2.000736
192,Realised Volatility Only,10.0,0.185107,0.160838,0.024269,0.248639,0.724266,-0.458528,0.082102,0.410509,2.000736
193,Realised Volatility Only,20.0,0.185107,0.137054,0.048053,0.248723,0.640840,-0.468790,0.082102,0.410509,2.000736
194,Realised Volatility Only,50.0,0.185107,0.068519,0.116588,0.249183,0.390567,-0.505420,0.082102,0.410509,2.000736


Offset-zero frequency/cost results: 10 trading day(s)


,portfolio,transaction_cost_bps,gross_annualised_return,net_annualised_return,annualised_return_cost_drag,net_annualised_volatility,net_sharpe,net_max_drawdown,average_daily_turnover,average_rebalance_turnover,average_gross_exposure
30,Momentum Only,0.0,0.020599,0.020599,0.000000,0.224191,0.203942,-0.481414,0.078885,0.790352,2.008225
31,Momentum Only,5.0,0.020599,0.010508,0.010091,0.224178,0.159617,-0.492782,0.078885,0.790352,2.008225
32,Momentum Only,10.0,0.020599,0.000513,0.020086,0.224181,0.115277,-0.507130,0.078885,0.790352,2.008225
33,Momentum Only,20.0,0.020599,-0.019195,0.039794,0.224241,0.026596,-0.550944,0.078885,0.790352,2.008225
34,Momentum Only,50.0,0.020599,-0.076116,0.096715,0.224836,-0.238722,-0.711422,0.078885,0.790352,2.008225
215,Realised Volatility Only,0.0,0.206899,0.206899,0.000000,0.246727,0.885669,-0.412695,0.062010,0.621283,2.001138
216,Realised Volatility Only,5.0,0.206899,0.197519,0.009379,0.246697,0.854103,-0.417205,0.062010,0.621283,2.001138
217,Realised Volatility Only,10.0,0.206899,0.188210,0.018689,0.246678,0.822494,-0.421681,0.062010,0.621283,2.001138
218,Realised Volatility Only,20.0,0.206899,0.169797,0.037102,0.246672,0.759165,-0.430535,0.062010,0.621283,2.001138
219,Realised Volatility Only,50.0,0.206899,0.116177,0.090722,0.246907,0.568574,-0.461983,0.062010,0.621283,2.001138


Offset-zero frequency/cost results: 21 trading day(s)


,portfolio,transaction_cost_bps,gross_annualised_return,net_annualised_return,annualised_return_cost_drag,net_annualised_volatility,net_sharpe,net_max_drawdown,average_daily_turnover,average_rebalance_turnover,average_gross_exposure
80,Momentum Only,0.0,0.013759,0.013759,0.000000,0.224692,0.174264,-0.510893,0.053710,1.132217,2.018171
81,Momentum Only,5.0,0.013759,0.006905,0.006854,0.224764,0.144098,-0.520575,0.053710,1.132217,2.018171
82,Momentum Only,10.0,0.013759,0.000093,0.013666,0.224854,0.113943,-0.530073,0.053710,1.132217,2.018171
83,Momentum Only,20.0,0.013759,-0.013406,0.027165,0.225086,0.053693,-0.549834,0.053710,1.132217,2.018171
84,Momentum Only,50.0,0.013759,-0.052913,0.066672,0.226201,-0.126081,-0.655134,0.053710,1.132217,2.018171
265,Realised Volatility Only,0.0,0.189139,0.189139,0.000000,0.238193,0.846643,-0.404171,0.045599,0.961229,2.003574
266,Realised Volatility Only,5.0,0.189139,0.182339,0.006800,0.238163,0.822625,-0.407192,0.045599,0.961229,2.003574
267,Realised Volatility Only,10.0,0.189139,0.175574,0.013565,0.238145,0.798560,-0.410199,0.045599,0.961229,2.003574
268,Realised Volatility Only,20.0,0.189139,0.162149,0.026990,0.238147,0.750303,-0.416171,0.045599,0.961229,2.003574
269,Realised Volatility Only,50.0,0.189139,0.122707,0.066432,0.238445,0.604791,-0.433758,0.045599,0.961229,2.003574


In [12]:
phase_cost_summary = (
    robustness_grid.groupby(
        ["portfolio", "rebalance_frequency", "transaction_cost_bps"],
        sort=False,
    )
    .agg(
        offset_count=("rebalance_offset", "nunique"),
        phase_mean_net_annualised_return=("net_annualised_return", "mean"),
        phase_median_net_annualised_return=("net_annualised_return", "median"),
        phase_minimum_net_annualised_return=("net_annualised_return", "min"),
        phase_mean_net_sharpe=("net_sharpe", "mean"),
        phase_median_net_sharpe=("net_sharpe", "median"),
        phase_worst_net_sharpe=("net_sharpe", "min"),
        phase_worst_max_drawdown=("net_max_drawdown", "min"),
        phase_mean_daily_turnover=("average_daily_turnover", "mean"),
        phase_mean_gross_exposure=("average_gross_exposure", "mean"),
        every_phase_profitable=(
            "net_annualised_return",
            lambda values: values.gt(0.0).all(),
        ),
    )
    .reset_index()
)
if not phase_cost_summary["offset_count"].eq(
    phase_cost_summary["rebalance_frequency"]
).all():
    raise ValueError("The phase-cost summary is incomplete.")

strategy_colours = {
    "Momentum Only": "#7C3AED",
    "Realised Volatility Only": "#DB2777",
    **{
        portfolio: PORTFOLIO_COLOURS[portfolio]
        for portfolio in ACTIVE_PORTFOLIO_ORDER[2:]
    },
}
cost_figure_data = phase_cost_summary.assign(
    frequency_label=lambda data: data["rebalance_frequency"].map(
        lambda value: f"{value} day(s)"
    )
)
cost_return_figure = px.line(
    cost_figure_data,
    x="transaction_cost_bps",
    y="phase_median_net_annualised_return",
    color="portfolio",
    facet_col="frequency_label",
    facet_col_wrap=2,
    markers=True,
    category_orders={"portfolio": list(ACTIVE_PORTFOLIO_ORDER)},
    color_discrete_map=strategy_colours,
    title="Phase-Median Net Return by Frequency and Cost",
)
cost_return_figure.update_layout(
    template="plotly_white",
    height=720,
    margin={"l": 64, "r": 24, "t": 96, "b": 56},
    legend={"orientation": "h", "y": 1.10, "x": 0.0},
)
cost_return_figure.update_xaxes(title_text="Transaction cost (bps)", showgrid=False)
cost_return_figure.update_yaxes(
    title_text="Median net annualised return",
    tickformat=".0%",
    gridcolor="#E2E8F0",
    zeroline=True,
)
for annotation in cost_return_figure.layout.annotations:
    annotation.text = annotation.text.split("=")[-1]
display(cost_return_figure)

### Frequency and cost findings

- **All 925 variants pass the construction and accounting audits.** Gross returns, turnover, and exposure remain unchanged across costs, while net return declines monotonically.
- **Realised Volatility has the highest offset-zero net return at every frequency under 10 bps:** 14.21%, 16.08%, 18.82%, and 17.56% at 1, 5, 10, and 21 days.
- **Slower rebalancing materially improves cost resilience.** At 50 bps, Realised Volatility remains positive at 5, 10, and 21 days; Momentum is negative at every frequency.
- **Composite's risk-adjusted result catches up at the slowest schedule.** At 21 days and 10 bps, its 0.813 offset-zero Sharpe exceeds Realised Volatility's 0.799 despite lower return.

The return advantage of Realised Volatility is not confined to one cost assumption, but its risk trade-off still prevents a return-only conclusion.

## 5. Rebalance-phase robustness

Phase evidence uses the baseline 10 bps cost and all valid offsets. Offset-zero tables preserve the original named implementations, while phase distributions receive greater weight in the interpretation.

In [13]:
baseline_phase_detail = robustness_grid.loc[
    robustness_grid["transaction_cost_bps"].eq(BASELINE_TRANSACTION_COST_BPS)
].copy()
phase_summary = (
    baseline_phase_detail.groupby(
        ["portfolio", "rebalance_frequency"],
        sort=False,
    )
    .agg(
        offset_count=("rebalance_offset", "nunique"),
        minimum_offset=("rebalance_offset", "min"),
        maximum_offset=("rebalance_offset", "max"),
        mean_net_annualised_return=("net_annualised_return", "mean"),
        median_net_annualised_return=("net_annualised_return", "median"),
        minimum_net_annualised_return=("net_annualised_return", "min"),
        maximum_net_annualised_return=("net_annualised_return", "max"),
        mean_net_sharpe=("net_sharpe", "mean"),
        median_net_sharpe=("net_sharpe", "median"),
        minimum_net_sharpe=("net_sharpe", "min"),
        maximum_net_sharpe=("net_sharpe", "max"),
        worst_max_drawdown=("net_max_drawdown", "min"),
        mean_daily_turnover=("average_daily_turnover", "mean"),
        mean_rebalance_turnover=("average_rebalance_turnover", "mean"),
        every_phase_profitable=(
            "net_annualised_return",
            lambda values: values.gt(0.0).all(),
        ),
    )
    .reset_index()
)
phase_summary["annualised_return_range"] = (
    phase_summary["maximum_net_annualised_return"]
    - phase_summary["minimum_net_annualised_return"]
)
phase_summary["audit_passes"] = (
    phase_summary["offset_count"].eq(phase_summary["rebalance_frequency"])
    & phase_summary["minimum_offset"].eq(0)
    & phase_summary["maximum_offset"].eq(
        phase_summary["rebalance_frequency"] - 1
    )
)
if len(baseline_phase_detail) != len(ACTIVE_PORTFOLIO_ORDER) * sum(
    ROBUSTNESS_REBALANCE_FREQUENCIES
):
    raise ValueError("The baseline phase detail is incomplete.")
if not phase_summary["audit_passes"].all():
    raise ValueError("The rebalance-phase summary audit failed.")

display(phase_summary.round(6))

,portfolio,rebalance_frequency,offset_count,minimum_offset,maximum_offset,mean_net_annualised_return,median_net_annualised_return,minimum_net_annualised_return,maximum_net_annualised_return,mean_net_sharpe,median_net_sharpe,minimum_net_sharpe,maximum_net_sharpe,worst_max_drawdown,mean_daily_turnover,mean_rebalance_turnover,every_phase_profitable,annualised_return_range,audit_passes
0,Momentum Only,1,1,0,0,-0.013606,-0.013606,-0.013606,-0.013606,0.049984,0.049984,0.049984,0.049984,-0.559783,0.253982,0.253982,False,0.000000,True
1,Momentum Only,5,5,0,4,0.009275,0.009488,0.001315,0.015548,0.153612,0.154481,0.118273,0.181720,-0.535324,0.111219,0.556094,True,0.014232,True
2,Momentum Only,10,10,0,9,0.010728,0.011094,-0.001619,0.024210,0.160437,0.162139,0.105255,0.219883,-0.553886,0.078034,0.780334,False,0.025829,True
3,Momentum Only,21,21,0,20,0.006262,0.004227,-0.006419,0.024075,0.141232,0.132248,0.084821,0.219106,-0.572786,0.052708,1.106893,False,0.030494,True
4,Realised Volatility Only,1,1,0,0,0.142083,0.142083,0.142083,0.142083,0.656169,0.656169,0.656169,0.656169,-0.448455,0.159444,0.159444,True,0.000000,True
5,Realised Volatility Only,5,5,0,4,0.164741,0.160991,0.159226,0.172984,0.737174,0.724931,0.718478,0.762086,-0.458528,0.082694,0.413470,True,0.013758,True
6,Realised Volatility Only,10,10,0,9,0.172279,0.168231,0.158525,0.191733,0.768152,0.754317,0.720741,0.837054,-0.467144,0.061663,0.616632,True,0.033208,True
7,Realised Volatility Only,21,21,0,20,0.170351,0.170961,0.149649,0.187990,0.768006,0.769909,0.695003,0.824365,-0.441129,0.045399,0.953376,True,0.038341,True
8,Composite Score,1,1,0,0,0.101512,0.101512,0.101512,0.101512,0.561033,0.561033,0.561033,0.561033,-0.329433,0.235922,0.235922,True,0.000000,True
9,Composite Score,5,5,0,4,0.130262,0.132628,0.114814,0.139830,0.681883,0.695545,0.619710,0.718364,-0.373850,0.105523,0.527615,True,0.025017,True


In [14]:
matched_frequency_rows = []
matched_frequency_sets = {
    "Five-day common baseline": (5, ACTIVE_PORTFOLIO_ORDER),
    "Ten-day sleeve comparison": (
        10,
        (
            "Momentum Only",
            "Realised Volatility Only",
            "Fixed 50/50 Sleeves",
            "Pure Inverse Volatility",
        ),
    ),
    "Twenty-one-day composite comparison": (
        21,
        ("Momentum Only", "Realised Volatility Only", "Composite Score"),
    ),
}
for comparison, (frequency, portfolios) in matched_frequency_sets.items():
    for portfolio in portfolios:
        matched_frequency_rows.append(
            {
                "comparison": comparison,
                "rebalance_frequency": frequency,
                "portfolio": portfolio,
            }
        )
matched_frequency_keys = pd.DataFrame(matched_frequency_rows)

matched_offset_zero = matched_frequency_keys.merge(
    baseline_phase_detail.loc[
        baseline_phase_detail["rebalance_offset"].eq(0),
        [
            "portfolio",
            "rebalance_frequency",
            "net_annualised_return",
            "net_annualised_volatility",
            "net_sharpe",
            "net_max_drawdown",
            "average_daily_turnover",
            "average_gross_exposure",
        ],
    ],
    on=["portfolio", "rebalance_frequency"],
    how="left",
    validate="one_to_one",
)
matched_phase_averages = matched_frequency_keys.merge(
    phase_summary[
        [
            "portfolio",
            "rebalance_frequency",
            "mean_net_annualised_return",
            "median_net_annualised_return",
            "minimum_net_annualised_return",
            "maximum_net_annualised_return",
            "mean_net_sharpe",
            "minimum_net_sharpe",
            "worst_max_drawdown",
            "mean_daily_turnover",
            "every_phase_profitable",
        ]
    ],
    on=["portfolio", "rebalance_frequency"],
    how="left",
    validate="one_to_one",
)
if matched_offset_zero.isna().any().any() or matched_phase_averages.isna().any().any():
    raise ValueError("A matched-frequency comparison is incomplete.")

print("Matched offset-zero implementations")
display(matched_offset_zero.round(6))
print("Matched phase-averaged implementations")
display(matched_phase_averages.round(6))

Matched offset-zero implementations


,comparison,rebalance_frequency,portfolio,net_annualised_return,net_annualised_volatility,net_sharpe,net_max_drawdown,average_daily_turnover,average_gross_exposure
0,Five-day common baseline,5,Momentum Only,0.009488,0.221888,0.154481,-0.509550,0.111424,2.002969
1,Five-day common baseline,5,Realised Volatility Only,0.160838,0.248639,0.724266,-0.458528,0.082102,2.000736
2,Five-day common baseline,5,Composite Score,0.137370,0.213330,0.710582,-0.306277,0.104652,2.000719
3,Five-day common baseline,5,Fixed 50/50 Sleeves,0.103533,0.168423,0.669545,-0.256562,0.087102,1.541543
4,Five-day common baseline,5,Pure Inverse Volatility,0.104115,0.162893,0.689863,-0.195652,0.092385,1.607862
5,Ten-day sleeve comparison,10,Momentum Only,0.000513,0.224181,0.115277,-0.507130,0.078885,2.008225
6,Ten-day sleeve comparison,10,Realised Volatility Only,0.188210,0.246678,0.822494,-0.421681,0.062010,2.001138
7,Ten-day sleeve comparison,10,Fixed 50/50 Sleeves,0.108436,0.167656,0.698280,-0.256835,0.062359,1.543356
8,Ten-day sleeve comparison,10,Pure Inverse Volatility,0.109593,0.162527,0.721505,-0.217693,0.066040,1.607989
9,Twenty-one-day composite comparison,21,Momentum Only,0.000093,0.224854,0.113943,-0.530073,0.053710,2.018171


Matched phase-averaged implementations


,comparison,rebalance_frequency,portfolio,mean_net_annualised_return,median_net_annualised_return,minimum_net_annualised_return,maximum_net_annualised_return,mean_net_sharpe,minimum_net_sharpe,worst_max_drawdown,mean_daily_turnover,every_phase_profitable
0,Five-day common baseline,5,Momentum Only,0.009275,0.009488,0.001315,0.015548,0.153612,0.118273,-0.535324,0.111219,True
1,Five-day common baseline,5,Realised Volatility Only,0.164741,0.160991,0.159226,0.172984,0.737174,0.718478,-0.458528,0.082694,True
2,Five-day common baseline,5,Composite Score,0.130262,0.132628,0.114814,0.139830,0.681883,0.619710,-0.373850,0.105523,True
3,Five-day common baseline,5,Fixed 50/50 Sleeves,0.101998,0.102375,0.099454,0.103607,0.661368,0.648117,-0.256562,0.087462,True
4,Five-day common baseline,5,Pure Inverse Volatility,0.104751,0.104115,0.102976,0.108525,0.693759,0.680305,-0.223379,0.092673,True
5,Ten-day sleeve comparison,10,Momentum Only,0.010728,0.011094,-0.001619,0.024210,0.160437,0.105255,-0.553886,0.078034,False
6,Ten-day sleeve comparison,10,Realised Volatility Only,0.172279,0.168231,0.158525,0.191733,0.768152,0.720741,-0.467144,0.061663,True
7,Ten-day sleeve comparison,10,Fixed 50/50 Sleeves,0.105018,0.104692,0.094368,0.113055,0.679803,0.623846,-0.275165,0.061722,True
8,Ten-day sleeve comparison,10,Pure Inverse Volatility,0.110999,0.111346,0.098314,0.119114,0.729192,0.660818,-0.235311,0.065537,True
9,Twenty-one-day composite comparison,21,Momentum Only,0.006262,0.004227,-0.006419,0.024075,0.141232,0.084821,-0.572786,0.052708,False


In [15]:
phase_figure_data = baseline_phase_detail.assign(
    frequency_label=lambda data: data["rebalance_frequency"].map(
        lambda value: f"{value} day(s)"
    )
)
phase_sharpe_figure = px.box(
    phase_figure_data,
    x="frequency_label",
    y="net_sharpe",
    color="portfolio",
    points="outliers",
    category_orders={
        "portfolio": list(ACTIVE_PORTFOLIO_ORDER),
        "frequency_label": [
            f"{frequency} day(s)"
            for frequency in ROBUSTNESS_REBALANCE_FREQUENCIES
        ],
    },
    color_discrete_map=strategy_colours,
    title="Net Sharpe Distribution Across Rebalance Phases – 10 bps",
)
phase_sharpe_figure.update_layout(
    template="plotly_white",
    height=520,
    margin={"l": 64, "r": 24, "t": 88, "b": 56},
    legend={"orientation": "h", "y": 1.10, "x": 0.0},
)
phase_sharpe_figure.update_xaxes(title_text="Rebalance frequency", showgrid=False)
phase_sharpe_figure.update_yaxes(
    title_text="Net Sharpe ratio",
    gridcolor="#E2E8F0",
    zeroline=True,
)
display(phase_sharpe_figure)

### Rebalance-phase findings

- **Realised Volatility leads phase-mean return at every frequency.** At the matched 21-day schedule it returns 17.04% on average, versus 15.84% for Composite.
- **Composite remains stronger on matched 21-day risk evidence:** phase-mean Sharpe is 0.802 versus 0.768, and its worst drawdown is -39.24% versus -44.11% for Realised Volatility. The pre-registered superiority expectation therefore requires qualification rather than simple confirmation.
- **The sleeve portfolios buy meaningful downside protection.** At 10 days, Realised Volatility averages 17.23% return with a -46.71% worst drawdown; Fixed 50/50 and Pure Inverse Volatility average 10.50% and 11.10%, with worst drawdowns of -27.52% and -23.53%.
- **Pure Inverse Volatility is the stronger matched sleeve blend:** at 10 days it exceeds Fixed 50/50 in phase-mean return and Sharpe while retaining the shallower worst drawdown.
- **Momentum remains a weak standalone benchmark.** Its 10- and 21-day phase sets include losing implementations, while every phase of the other four strategies is profitable at 10 bps.

Phase robustness confirms genuine return strength in Realised Volatility, but also confirms that the multi-factor portfolios provide material risk reduction. Subperiod and attribution evidence are still required before reassessing the final hierarchy.

## 6. Subperiod and rolling stability

The controlled five-day baseline is evaluated over the final research subperiods: **2016–2018**, **2019–2022**, and **2023–present**. Rolling evidence uses complete 252-day windows. The lower tail is defined before calculation as the 10th percentile; for volatility, the adverse tail is the 90th percentile. SPY remains contextual and carries no simulated transaction costs.

In [16]:
SUBPERIODS = {
    "2016–2018": ("2016-01-07", "2018-12-31"),
    "2019–2022": ("2019-01-01", "2022-12-31"),
    "2023–present": ("2023-01-01", str(CHALLENGE_END_DATE.date())),
}
SUBPERIOD_ORDER = tuple(SUBPERIODS)
subperiod_parts = []

for portfolio in ACTIVE_PORTFOLIO_ORDER:
    summary = summarise_backtest_subperiods(
        common_daily[portfolio],
        periods=SUBPERIODS,
        return_column="net_return",
    )
    summary["portfolio"] = portfolio
    subperiod_parts.append(summary)

spy_subperiod_rows = []
for period, (start_date, end_date) in SUBPERIODS.items():
    period_returns = benchmark_daily.loc[
        benchmark_daily["date"].between(start_date, end_date),
        "benchmark_return",
    ]
    performance = summarise_returns(period_returns)
    spy_subperiod_rows.append(
        {
            "portfolio": "SPY",
            "period": period,
            "observations": performance["observations"],
            "total_return": performance["total_return"],
            "annualised_return": performance["annualised_return"],
            "annualised_volatility": performance["annualised_volatility"],
            "sharpe_ratio": performance["sharpe_ratio"],
            "max_drawdown": performance["maximum_drawdown"],
            "positive_day_fraction": performance["positive_day_fraction"],
            "average_daily_turnover": 0.0,
        }
    )

subperiod_summary = pd.concat(
    [*subperiod_parts, pd.DataFrame(spy_subperiod_rows)],
    ignore_index=True,
)
portfolio_positions = {
    portfolio: position for position, portfolio in enumerate(COMPARISON_ORDER)
}
period_positions = {
    period: position for position, period in enumerate(SUBPERIOD_ORDER)
}
subperiod_summary = (
    subperiod_summary.assign(
        _portfolio_order=lambda data: data["portfolio"].map(portfolio_positions),
        _period_order=lambda data: data["period"].map(period_positions),
    )
    .sort_values(["_portfolio_order", "_period_order"], kind="stable")
    .drop(columns=["_portfolio_order", "_period_order"])
    .reset_index(drop=True)
)

subperiod_audit = (
    subperiod_summary.groupby("period", sort=False)
    .agg(
        portfolio_count=("portfolio", "nunique"),
        minimum_observations=("observations", "min"),
        maximum_observations=("observations", "max"),
        missing_metrics=(
            "annualised_return",
            lambda values: values.isna().sum(),
        ),
    )
    .reset_index()
)
subperiod_audit["audit_passes"] = (
    subperiod_audit["portfolio_count"].eq(len(COMPARISON_ORDER))
    & subperiod_audit["minimum_observations"].eq(
        subperiod_audit["maximum_observations"]
    )
    & subperiod_audit["missing_metrics"].eq(0)
)
if len(subperiod_summary) != len(COMPARISON_ORDER) * len(SUBPERIOD_ORDER):
    raise ValueError("The subperiod summary is incomplete.")
if subperiod_summary.duplicated(["portfolio", "period"]).any():
    raise ValueError("The subperiod summary contains duplicate keys.")
if not subperiod_audit["audit_passes"].all():
    raise ValueError("The subperiod audit failed.")

display(subperiod_audit)
display(
    subperiod_summary[
        [
            "portfolio",
            "period",
            "observations",
            "annualised_return",
            "annualised_volatility",
            "sharpe_ratio",
            "max_drawdown",
            "positive_day_fraction",
            "average_daily_turnover",
        ]
    ].round(6)
)

,period,portfolio_count,minimum_observations,maximum_observations,missing_metrics,audit_passes
0,2016–2018,6,751.0,751.0,0,True
1,2019–2022,6,1008.0,1008.0,0,True
2,2023–present,6,876.0,876.0,0,True


,portfolio,period,observations,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown,positive_day_fraction,average_daily_turnover
0,Momentum Only,2016–2018,751.0,-0.006414,0.159625,0.039761,-0.228463,0.543276,0.104216
1,Momentum Only,2019–2022,1008.0,-0.043029,0.244072,-0.056544,-0.406332,0.531746,0.113771
2,Momentum Only,2023–present,876.0,0.088206,0.240361,0.472397,-0.285738,0.538813,0.114901
3,Realised Volatility Only,2016–2018,751.0,0.059451,0.171072,0.423465,-0.250825,0.539281,0.088488
4,Realised Volatility Only,2019–2022,1008.0,0.006100,0.283471,0.162723,-0.458528,0.496032,0.078284
5,Realised Volatility Only,2023–present,876.0,0.480100,0.261104,1.633445,-0.190333,0.563927,0.081020
6,Composite Score,2016–2018,751.0,0.046026,0.168399,0.351875,-0.266899,0.556591,0.102925
7,Composite Score,2019–2022,1008.0,0.015134,0.201875,0.175423,-0.306277,0.527778,0.107657
8,Composite Score,2023–present,876.0,0.392808,0.255964,1.423740,-0.226851,0.576484,0.102675
9,Fixed 50/50 Sleeves,2016–2018,751.0,0.032319,0.136945,0.300987,-0.195185,0.549933,0.089042


In [17]:
post_2022_dependence_rows = []
for portfolio in COMPARISON_ORDER:
    portfolio_periods = subperiod_summary.loc[
        subperiod_summary["portfolio"].eq(portfolio)
    ].set_index("period")
    prior = portfolio_periods.loc[list(SUBPERIOD_ORDER[:2])]
    recent = portfolio_periods.loc[SUBPERIOD_ORDER[-1]]
    post_2022_dependence_rows.append(
        {
            "portfolio": portfolio,
            "prior_period_mean_annualised_return": prior["annualised_return"].mean(),
            "post_2022_annualised_return": recent["annualised_return"],
            "post_2022_minus_prior_mean_return": (
                recent["annualised_return"] - prior["annualised_return"].mean()
            ),
            "prior_period_mean_sharpe": prior["sharpe_ratio"].mean(),
            "post_2022_sharpe": recent["sharpe_ratio"],
            "post_2022_minus_prior_mean_sharpe": (
                recent["sharpe_ratio"] - prior["sharpe_ratio"].mean()
            ),
        }
    )
post_2022_dependence = pd.DataFrame(post_2022_dependence_rows)
display(post_2022_dependence.round(6))

,portfolio,prior_period_mean_annualised_return,post_2022_annualised_return,post_2022_minus_prior_mean_return,prior_period_mean_sharpe,post_2022_sharpe,post_2022_minus_prior_mean_sharpe
0,Momentum Only,-0.024721,0.088206,0.112928,-0.008391,0.472397,0.480788
1,Realised Volatility Only,0.032776,0.480100,0.447325,0.293094,1.633445,1.340351
2,Composite Score,0.030580,0.392808,0.362228,0.263649,1.423740,1.160091
3,Fixed 50/50 Sleeves,0.022444,0.290069,0.267625,0.229630,1.347022,1.117391
4,Pure Inverse Volatility,0.032982,0.259435,0.226453,0.302129,1.243605,0.941476
5,SPY,0.120142,0.227107,0.106966,0.767042,1.424677,0.657634


In [18]:
rolling_state_252 = calculate_performance_risk_state(
    common_baseline_daily[["date", "portfolio", "net_return"]],
    benchmark_daily[["date", "benchmark_return"]],
    portfolios=ACTIVE_PORTFOLIO_ORDER,
    performance_window=252,
    risk_window=252,
)
rolling_metric_columns = [
    "trailing_return_252",
    "annualised_volatility_252",
    "rolling_sharpe_252",
    "maximum_drawdown_252",
]
rolling_summary_rows = []
rolling_audit_rows = []
expected_complete_windows = len(common_dates) - 252 + 1

for portfolio in COMPARISON_ORDER:
    portfolio_state = rolling_state_252.loc[
        rolling_state_252["portfolio"].eq(portfolio)
    ].sort_values("date")
    complete = portfolio_state.dropna(subset=rolling_metric_columns)
    rolling_audit_rows.append(
        {
            "portfolio": portfolio,
            "observations": len(portfolio_state),
            "complete_windows": len(complete),
            "first_complete_date": complete["date"].min(),
            "last_complete_date": complete["date"].max(),
            "audit_passes": bool(
                len(portfolio_state) == len(common_dates)
                and len(complete) == expected_complete_windows
                and complete["date"].max() == common_dates.max()
            ),
        }
    )
    rolling_summary_rows.append(
        {
            "portfolio": portfolio,
            "complete_windows": len(complete),
            "median_trailing_return_252": complete["trailing_return_252"].median(),
            "p10_trailing_return_252": complete["trailing_return_252"].quantile(0.10),
            "minimum_trailing_return_252": complete["trailing_return_252"].min(),
            "positive_trailing_return_fraction": complete["trailing_return_252"].gt(0.0).mean(),
            "median_rolling_sharpe_252": complete["rolling_sharpe_252"].median(),
            "p10_rolling_sharpe_252": complete["rolling_sharpe_252"].quantile(0.10),
            "minimum_rolling_sharpe_252": complete["rolling_sharpe_252"].min(),
            "median_annualised_volatility_252": complete["annualised_volatility_252"].median(),
            "p90_annualised_volatility_252": complete["annualised_volatility_252"].quantile(0.90),
            "maximum_annualised_volatility_252": complete["annualised_volatility_252"].max(),
            "median_maximum_drawdown_252": complete["maximum_drawdown_252"].median(),
            "p10_maximum_drawdown_252": complete["maximum_drawdown_252"].quantile(0.10),
            "minimum_maximum_drawdown_252": complete["maximum_drawdown_252"].min(),
        }
    )

rolling_audit = pd.DataFrame(rolling_audit_rows)
rolling_summary = pd.DataFrame(rolling_summary_rows)
if not rolling_audit["audit_passes"].all():
    raise ValueError("The rolling-window audit failed.")

display(rolling_audit)
display(rolling_summary.round(6))

,portfolio,observations,complete_windows,first_complete_date,last_complete_date,audit_passes
0,Momentum Only,2635,2384,2017-01-05,2026-07-01,True
1,Realised Volatility Only,2635,2384,2017-01-05,2026-07-01,True
2,Composite Score,2635,2384,2017-01-05,2026-07-01,True
3,Fixed 50/50 Sleeves,2635,2384,2017-01-05,2026-07-01,True
4,Pure Inverse Volatility,2635,2384,2017-01-05,2026-07-01,True
5,SPY,2635,2384,2017-01-05,2026-07-01,True


,portfolio,complete_windows,median_trailing_return_252,p10_trailing_return_252,minimum_trailing_return_252,positive_trailing_return_fraction,median_rolling_sharpe_252,p10_rolling_sharpe_252,minimum_rolling_sharpe_252,median_annualised_volatility_252,p90_annualised_volatility_252,maximum_annualised_volatility_252,median_maximum_drawdown_252,p10_maximum_drawdown_252,minimum_maximum_drawdown_252
0,Momentum Only,2384,0.004878,-0.213602,-0.356836,0.515101,0.124268,-0.874696,-1.462570,0.197120,0.293976,0.373130,-0.159124,-0.311169,-0.391405
1,Realised Volatility Only,2384,0.189653,-0.152963,-0.420746,0.757970,0.869629,-0.567085,-1.331372,0.227143,0.328284,0.375346,-0.185316,-0.320636,-0.438306
2,Composite Score,2384,0.137100,-0.145236,-0.260633,0.717282,0.844760,-0.688732,-1.332568,0.209894,0.254671,0.311637,-0.160275,-0.266899,-0.304446
3,Fixed 50/50 Sleeves,2384,0.104232,-0.103978,-0.197500,0.716862,0.798417,-0.633347,-1.408990,0.156688,0.204951,0.253783,-0.116049,-0.195185,-0.221119
4,Pure Inverse Volatility,2384,0.098701,-0.066061,-0.147357,0.749581,0.769788,-0.441884,-1.201598,0.147720,0.202204,0.254190,-0.102205,-0.171496,-0.195652
5,SPY,2384,0.168396,-0.045186,-0.197327,0.867450,1.033672,-0.123054,-0.790579,0.151572,0.291345,0.336464,-0.101019,-0.283191,-0.337173


In [19]:
display(
    build_rolling_metric_figure(
        rolling_state_252,
        "trailing_return_252",
        title="Trailing 252-Day Return – Five-Day Baseline",
    )
)
display(
    build_rolling_metric_figure(
        rolling_state_252,
        "rolling_sharpe_252",
        title="Rolling 252-Day Sharpe – Five-Day Baseline",
    )
)

### Subperiod and rolling findings

- **The full-sample results are strongly post-2022 dependent.** Realised Volatility returns 48.01% annually in 2023–present, versus a 3.28% mean across the two earlier subperiods; Composite rises from 3.06% to 39.28%.
- **Pre-2023 evidence is much weaker.** Realised Volatility returns 5.95% in 2016–2018 and 0.61% in 2019–2022. Pure Inverse Volatility is the strongest active strategy in 2019–2022, but returns only 4.06%.
- **Realised Volatility has the strongest active median rolling return:** 18.97%, versus 13.71% for Composite. Its 10th-percentile and minimum rolling returns are -15.30% and -42.07%, showing substantial left-tail instability.
- **Pure Inverse Volatility provides the best active lower-tail protection.** Its 10th-percentile rolling return is -6.61% and its worst rolling drawdown is -19.57%, compared with -15.30% and -43.83% for Realised Volatility.
- **SPY remains stronger on consistency:** 86.75% of its rolling annual returns are positive, compared with 75.80% for Realised Volatility and about 72% for Composite and Fixed 50/50.

The post-2022 regime materially drives the full-sample ranking. Realised Volatility's return advantage is genuine in that period, but it is not stable across the full research history.

## 7. Long/short attribution

Attribution uses the common five-day, 10 bps reconstruction. Long, short, cost, and net figures are arithmetic contributions so that the identities add exactly; they are not geometric headline returns. The all-portfolio table provides a controlled comparison, followed by standalone-factor subperiod detail.

In [20]:
side_cost_attribution = build_side_cost_attribution_summary(
    common_baseline_daily,
    portfolios=ACTIVE_PORTFOLIO_ORDER,
)
attribution_audit_rows = []
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_baseline_daily.loc[
        common_baseline_daily["portfolio"].eq(portfolio)
    ]
    summary = side_cost_attribution.loc[
        side_cost_attribution["portfolio"].eq(portfolio)
    ].iloc[0]
    maximum_daily_difference = max(
        (daily["long_return"] + daily["short_return"] - daily["gross_return"]).abs().max(),
        (daily["gross_return"] - daily["transaction_cost"] - daily["net_return"]).abs().max(),
    )
    maximum_summary_difference = max(
        abs(
            summary["annualised_long_contribution"]
            + summary["annualised_short_contribution"]
            - summary["annualised_gross_contribution"]
        ),
        abs(
            summary["annualised_gross_contribution"]
            - summary["annualised_cost_drag"]
            - summary["annualised_net_contribution"]
        ),
    )
    attribution_audit_rows.append(
        {
            "portfolio": portfolio,
            "maximum_daily_difference": maximum_daily_difference,
            "maximum_summary_difference": maximum_summary_difference,
            "audit_passes": bool(
                maximum_daily_difference < AUDIT_TOLERANCE
                and maximum_summary_difference < AUDIT_TOLERANCE
            ),
        }
    )
attribution_audit = pd.DataFrame(attribution_audit_rows)
if not attribution_audit["audit_passes"].all():
    raise ValueError("The side/cost attribution does not reconcile.")

side_cost_attribution_display = side_cost_attribution.copy()
attribution_numeric_columns = side_cost_attribution_display.select_dtypes(
    include="number"
).columns
side_cost_attribution_display[attribution_numeric_columns] = (
    side_cost_attribution_display[attribution_numeric_columns].round(6)
)
display(attribution_audit)
display(side_cost_attribution_display)

,portfolio,maximum_daily_difference,maximum_summary_difference,audit_passes
0,Momentum Only,0.0,5.551115e-17,True
1,Realised Volatility Only,0.0,5.551115e-17,True
2,Composite Score,0.0,2.775558e-17,True
3,Fixed 50/50 Sleeves,0.0,2.775558e-17,True
4,Pure Inverse Volatility,0.0,1.387779e-17,True


,portfolio,observations,rebalance_count,start_date,end_date,cumulative_long_contribution,cumulative_short_contribution,cumulative_gross_contribution,cumulative_transaction_cost,cumulative_net_contribution,annualised_long_contribution,annualised_short_contribution,annualised_gross_contribution,annualised_cost_drag,annualised_net_contribution,average_daily_turnover,average_rebalance_turnover
0,Momentum Only,2635,527,2016-01-07,2026-07-01,2.782073,-2.130053,0.652020,0.293602,0.358418,0.266065,-0.203709,0.062356,0.028079,0.034278,0.111424,0.557119
1,Realised Volatility Only,2635,527,2016-01-07,2026-07-01,3.431350,-1.332020,2.099330,0.216338,1.882992,0.328159,-0.127389,0.200771,0.020690,0.180081,0.082102,0.410509
2,Composite Score,2635,527,2016-01-07,2026-07-01,3.266775,-1.405957,1.860818,0.275758,1.585060,0.312420,-0.134460,0.177961,0.026372,0.151588,0.104652,0.523261
3,Fixed 50/50 Sleeves,2635,527,2016-01-07,2026-07-01,2.471070,-1.062427,1.408643,0.229513,1.179130,0.236322,-0.101606,0.134717,0.021950,0.112767,0.087102,0.435508
4,Pure Inverse Volatility,2635,527,2016-01-07,2026-07-01,2.585813,-1.167356,1.418457,0.243433,1.175024,0.247296,-0.111641,0.135655,0.023281,0.112374,0.092385,0.461923


In [21]:
factor_subperiod_attribution_parts = []
for period, (start_date, end_date) in SUBPERIODS.items():
    period_summary = build_side_cost_attribution_summary(
        common_baseline_daily,
        portfolios=ACTIVE_PORTFOLIO_ORDER[:2],
        start_date=start_date,
        end_date=end_date,
    )
    period_summary.insert(1, "period", period)
    factor_subperiod_attribution_parts.append(period_summary)
factor_subperiod_attribution = pd.concat(
    factor_subperiod_attribution_parts,
    ignore_index=True,
)
factor_subperiod_attribution["summary_difference"] = (
    factor_subperiod_attribution["annualised_long_contribution"]
    + factor_subperiod_attribution["annualised_short_contribution"]
    - factor_subperiod_attribution["annualised_cost_drag"]
    - factor_subperiod_attribution["annualised_net_contribution"]
).abs()
if len(factor_subperiod_attribution) != len(ACTIVE_PORTFOLIO_ORDER[:2]) * len(
    SUBPERIOD_ORDER
):
    raise ValueError("The standalone-factor subperiod attribution is incomplete.")
if factor_subperiod_attribution["summary_difference"].ge(AUDIT_TOLERANCE).any():
    raise ValueError("The standalone-factor subperiod attribution does not reconcile.")

display(
    factor_subperiod_attribution[
        [
            "portfolio",
            "period",
            "annualised_long_contribution",
            "annualised_short_contribution",
            "annualised_gross_contribution",
            "annualised_cost_drag",
            "annualised_net_contribution",
            "summary_difference",
        ]
    ].round(6)
)

,portfolio,period,annualised_long_contribution,annualised_short_contribution,annualised_gross_contribution,annualised_cost_drag,annualised_net_contribution,summary_difference
0,Momentum Only,2016–2018,0.215924,-0.183315,0.032609,0.026262,0.006347,0.0
1,Realised Volatility Only,2016–2018,0.241633,-0.146891,0.094742,0.022299,0.072443,0.0
2,Momentum Only,2019–2022,0.216866,-0.201997,0.014870,0.028670,-0.013801,0.0
3,Realised Volatility Only,2019–2022,0.227425,-0.161570,0.065855,0.019728,0.046127,0.0
4,Momentum Only,2023–present,0.365664,-0.223164,0.142501,0.028955,0.113546,0.0
5,Realised Volatility Only,2023–present,0.518253,-0.071337,0.446916,0.020417,0.426499,0.0


In [22]:
display(
    build_side_cost_attribution_figure(
        side_cost_attribution,
        title="Annualised Long, Short, and Cost Contributions – Five-Day Baseline",
    )
)

### Long/short attribution findings

- **Realised Volatility is long-side dominated, not broad across both legs.** Its annualised arithmetic contributions are +32.82% from longs, -12.74% from shorts, and -2.07% from costs, leaving +18.01% net.
- **The short side detracts in every subperiod.** Realised Volatility's short contribution is -14.69% in 2016–2018, -16.16% in 2019–2022, and -7.13% in 2023–present. Recent improvement combines a much stronger long side with a less negative short side.
- **Momentum's weak standalone result reflects short-side drag and costs:** +26.61% long, -20.37% short, and -2.81% cost produce only +3.43% annualised net contribution. Its 2019–2022 net contribution is negative.
- **Long-side dominance is shared by the candidates.** Composite records +31.24% long, -13.45% short, and -2.64% cost contributions; neither the factor nor candidate evidence establishes a profitable short book.
- **All identities reconcile.** Daily and annualised long-plus-short-to-gross and gross-minus-cost-to-net differences are below numerical tolerance.

The attribution weakens any claim that Realised Volatility represents balanced long-short alpha. Its advantage is primarily a strong long ranking, concentrated in the latest period.

## 8. Risk and concentration

The controlled five-day holdings are now evaluated with the repository's existing beta and concentration diagnostics. Realised beta uses a trailing 126-day window; return-contribution concentration uses 63 days.

In [23]:
common_security_holding_parts = []
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    holdings = common_holdings_full[portfolio].loc[
        common_holdings_full[portfolio]["date"].isin(common_dates)
    ].copy()
    holdings = holdings.assign(
        portfolio=portfolio,
        rebalance_frequency=COMMON_BASELINE_CONFIG.rebalance_frequency,
        rebalance_offset=COMMON_BASELINE_CONFIG.rebalance_offset,
        role="Controlled benchmark",
    )
    common_security_holding_parts.append(holdings)

common_security_holdings = (
    pd.concat(common_security_holding_parts, ignore_index=True)
    .sort_values(["portfolio", "date", "ticker"])
    .reset_index(drop=True)
)
common_security_daily = prepare_security_attribution(
    common_security_holdings,
    return_panel,
    transaction_cost_bps=BASELINE_TRANSACTION_COST_BPS,
)
security_attribution_audit = reconcile_security_attribution(
    common_baseline_daily,
    common_security_daily,
)
if not security_attribution_audit["audit_passes"].all():
    raise ValueError("The common security attribution does not reconcile.")

display(security_attribution_audit)

,portfolio,observations,max_abs_long_return_difference,max_abs_short_return_difference,max_abs_gross_return_difference,max_abs_turnover_difference,max_abs_transaction_cost_difference,max_abs_net_return_difference,max_abs_long_exposure_difference,max_abs_short_exposure_difference,max_abs_missing_return_weight_difference,maximum_absolute_difference,audit_passes
0,Composite Score,2635,1.387779e-17,6.938894e-18,1.387779e-17,2.220446e-16,2.168404e-19,1.387779e-17,2.220446e-16,2.220446e-16,0.0,2.220446e-16,True
1,Fixed 50/50 Sleeves,2635,1.387779e-17,1.387779e-17,2.775558e-17,2.220446e-16,2.168404e-19,2.775558e-17,2.220446e-16,2.220446e-16,0.0,2.220446e-16,True
2,Momentum Only,2635,1.387779e-17,1.387779e-17,2.775558e-17,2.220446e-16,2.168404e-19,2.775558e-17,2.220446e-16,2.220446e-16,0.0,2.220446e-16,True
3,Pure Inverse Volatility,2635,1.387779e-17,1.387779e-17,1.908196e-17,2.220446e-16,2.168404e-19,1.908196e-17,2.220446e-16,2.220446e-16,0.0,2.220446e-16,True
4,Realised Volatility Only,2635,2.775558e-17,6.938894e-18,2.775558e-17,2.220446e-16,2.168404e-19,2.775558e-17,2.220446e-16,2.220446e-16,0.0,2.220446e-16,True


In [24]:
common_holdings_beta_detail = prepare_holdings_beta_detail(
    common_security_holdings,
    factor_panel,
    portfolios=ACTIVE_PORTFOLIO_ORDER,
)
common_beta_state = calculate_beta_state(
    common_baseline_daily,
    benchmark_daily,
    common_security_holdings,
    factor_panel,
    portfolios=ACTIVE_PORTFOLIO_ORDER,
)
common_concentration_state = calculate_concentration_state(
    common_security_daily,
    factor_panel,
    common_holdings_beta_detail,
    portfolios=ACTIVE_PORTFOLIO_ORDER,
)

risk_diagnostic_audit = (
    common_beta_state.groupby("portfolio", sort=False)
    .agg(
        beta_dates=("date", "nunique"),
        minimum_beta_coverage=("beta_coverage", "min"),
    )
    .join(
        common_concentration_state.groupby("portfolio", sort=False).agg(
            concentration_dates=("date", "nunique")
        )
    )
    .reindex(ACTIVE_PORTFOLIO_ORDER)
    .reset_index()
)
risk_diagnostic_audit["audit_passes"] = (
    risk_diagnostic_audit["beta_dates"].eq(len(common_dates))
    & risk_diagnostic_audit["concentration_dates"].eq(len(common_dates))
)
if not risk_diagnostic_audit["audit_passes"].all():
    raise ValueError("The risk diagnostics do not cover the common window.")

display(risk_diagnostic_audit)

,portfolio,beta_dates,minimum_beta_coverage,concentration_dates,audit_passes
0,Momentum Only,2635,1.0,2635,True
1,Realised Volatility Only,2635,1.0,2635,True
2,Composite Score,2635,1.0,2635,True
3,Fixed 50/50 Sleeves,2635,1.0,2635,True
4,Pure Inverse Volatility,2635,1.0,2635,True


In [25]:
sector_counts = factor_panel.groupby("ticker")["sector"].nunique(dropna=False)
if sector_counts.gt(1).any():
    raise ValueError("Sector metadata are not stable by ticker.")
sector_metadata = factor_panel[["ticker", "sector"]].drop_duplicates("ticker")
sector_summary_parts = []
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    holdings = common_security_holdings.loc[
        common_security_holdings["portfolio"].eq(portfolio),
        ["date", "ticker", "weight"],
    ]
    exposure = calculate_sector_exposure(holdings, sector_metadata)
    summary = summarise_sector_exposure(exposure)
    average_absolute_net = (
        exposure.assign(absolute_net_weight=exposure["net_weight"].abs())
        .groupby("sector")["absolute_net_weight"]
        .mean()
        .rename("average_absolute_net_weight")
        .reset_index()
    )
    sector_summary_parts.append(
        summary.merge(average_absolute_net, on="sector", validate="one_to_one")
        .assign(portfolio=portfolio)
    )

sector_net_exposure_summary = pd.concat(sector_summary_parts, ignore_index=True)
average_sector_net_exposure = (
    sector_net_exposure_summary.pivot(
        index="sector", columns="portfolio", values="average_net_weight"
    )
    .reindex(columns=ACTIVE_PORTFOLIO_ORDER)
    .sort_index()
)
largest_sector_net_exposures = (
    sector_net_exposure_summary.sort_values(
        ["portfolio", "maximum_absolute_net_weight"],
        ascending=[True, False],
    )
    .groupby("portfolio", sort=False)
    .head(3)[
        [
            "portfolio",
            "sector",
            "average_net_weight",
            "average_absolute_net_weight",
            "maximum_absolute_net_weight",
        ]
    ]
)
display(average_sector_net_exposure.round(6))
display(largest_sector_net_exposures.round(6))

portfolio,Momentum Only,Realised Volatility Only,Composite Score,Fixed 50/50 Sleeves,Pure Inverse Volatility
sector,,,,,
Communication Services,-0.007609,0.008284,0.004832,0.000316,-0.001317
Consumer Discretionary,-0.006049,0.070233,0.050572,0.032088,0.028998
Consumer Staples,-0.041657,-0.277120,-0.212058,-0.159412,-0.147473
Energy,-0.019928,0.014481,-0.002194,-0.002671,-0.000448
Financials,-0.008201,-0.022258,-0.025444,-0.015137,-0.015343
Health Care,-0.046815,-0.113615,-0.111529,-0.080209,-0.073363
Industrials,-0.021190,0.033444,-0.001995,0.006204,0.000122
Information Technology,0.169576,0.400809,0.375137,0.285221,0.268019
Materials,0.001319,-0.016582,-0.012140,-0.007634,-0.006928


,portfolio,sector,average_net_weight,average_absolute_net_weight,maximum_absolute_net_weight
22,Composite Score,Information Technology,0.375137,0.375140,0.758488
26,Composite Score,Financials,-0.025444,0.145886,0.560104
24,Composite Score,Consumer Staples,-0.212058,0.215920,0.475202
33,Fixed 50/50 Sleeves,Information Technology,0.285221,0.285505,0.559852
35,Fixed 50/50 Sleeves,Consumer Staples,-0.159412,0.159513,0.378929
37,Fixed 50/50 Sleeves,Financials,-0.015137,0.093121,0.331482
0,Momentum Only,Information Technology,0.169576,0.240631,0.611316
3,Momentum Only,Financials,-0.008201,0.194038,0.560393
1,Momentum Only,Health Care,-0.046815,0.160310,0.450000
44,Pure Inverse Volatility,Information Technology,0.268019,0.269302,0.560625


In [26]:
beta_summary = (
    common_beta_state.groupby("portfolio", sort=False)
    .agg(
        average_beta_coverage=("beta_coverage", "mean"),
        average_holdings_market_beta=("holdings_market_beta", "mean"),
        average_realised_gross_beta_126=("realised_gross_beta_126", "mean"),
        average_beta_measurement_gap=("beta_measurement_gap", "mean"),
        average_absolute_beta_measurement_gap=(
            "beta_measurement_gap", lambda values: values.abs().mean()
        ),
    )
    .reset_index()
)
concentration_summary = (
    common_concentration_state.groupby("portfolio", sort=False)
    .agg(
        average_effective_position_count=("effective_position_count", "mean"),
        average_largest_absolute_sector_net_exposure=(
            "largest_absolute_sector_net_exposure", "mean"
        ),
        maximum_largest_absolute_sector_net_exposure=(
            "largest_absolute_sector_net_exposure", "max"
        ),
        average_effective_beta_contributor_count=(
            "effective_beta_contributor_count", "mean"
        ),
        average_top_five_beta_contribution_share=(
            "top_five_absolute_beta_contribution_share", "mean"
        ),
        average_effective_contributor_count_63=(
            "effective_contributor_count_63", "mean"
        ),
        average_top_five_return_contribution_share_63=(
            "top_five_contributor_share_63", "mean"
        ),
        average_effective_contribution_sector_count_63=(
            "effective_contribution_sector_count_63", "mean"
        ),
    )
    .reset_index()
)
risk_concentration_summary = (
    common_window_baseline[
        ["portfolio", "annualised_volatility", "max_drawdown"]
    ]
    .loc[lambda data: data["portfolio"].isin(ACTIVE_PORTFOLIO_ORDER)]
    .merge(beta_summary, on="portfolio", validate="one_to_one")
    .merge(concentration_summary, on="portfolio", validate="one_to_one")
    .set_index("portfolio")
    .reindex(ACTIVE_PORTFOLIO_ORDER)
    .reset_index()
)
display(risk_concentration_summary.round(6))

,portfolio,annualised_volatility,max_drawdown,average_beta_coverage,average_holdings_market_beta,average_realised_gross_beta_126,average_beta_measurement_gap,average_absolute_beta_measurement_gap,average_effective_position_count,average_largest_absolute_sector_net_exposure,maximum_largest_absolute_sector_net_exposure,average_effective_beta_contributor_count,average_top_five_beta_contribution_share,average_effective_contributor_count_63,average_top_five_return_contribution_share_63,average_effective_contribution_sector_count_63
0,Momentum Only,0.221888,-0.509550,1.0,0.260630,0.317779,-0.031505,0.247968,39.972592,0.328789,0.611316,31.706564,0.240055,41.943042,0.209475,5.687566
1,Realised Volatility Only,0.248639,-0.458528,1.0,0.962173,0.921344,0.052890,0.119733,39.972070,0.427686,0.809424,29.482534,0.256220,37.911373,0.236004,5.343012
2,Composite Score,0.213330,-0.306277,1.0,0.803854,0.782639,0.043964,0.136384,39.974818,0.395532,0.758488,30.412473,0.252272,40.426595,0.226709,5.436786
3,Fixed 50/50 Sleeves,0.168423,-0.256562,1.0,0.611724,0.619568,0.011072,0.122589,47.218563,0.298522,0.559852,31.796946,0.285189,41.710731,0.245314,5.415346
4,Pure Inverse Volatility,0.162893,-0.195652,1.0,0.567970,0.574195,0.011851,0.142923,49.889625,0.288783,0.560625,34.499842,0.269259,44.490227,0.231525,5.538485


### Risk and concentration findings

- **Realised Volatility carries the most market and sector risk.** Its average holdings beta is 0.962 (realised gross beta 0.921), versus 0.804 for Composite, 0.612 for Fixed, and 0.568 for Pure. Its average largest absolute sector net exposure is 42.77%, peaking at 80.94%.
- **Its sector tilts are economically large.** Realised Volatility averages +40.08% net Information Technology and -27.71% Consumer Staples; the comparable Composite tilts are +37.51% and -21.21%, while both sleeve portfolios are less imbalanced.
- **The return advantage comes with materially worse tail risk.** Realised Volatility records 24.86% volatility and a -45.85% drawdown. Composite reduces these to 21.33% and -30.63%; Fixed and Pure reduce them further to about 16% volatility and -25.66% and -19.57% drawdowns.
- **Composite does not add position breadth, but modestly diversifies realised contributions.** Both hold about 40 effective positions; Realised Volatility's top-five 63-day return-contribution share is 23.60% versus Composite's 22.67%, and it has the lowest effective contribution-sector count at 5.34. Fixed and Pure broaden holdings to 47.22 and 49.89 effective positions.
- **The signed beta gaps are small relative to the risk differences.** They average +0.053 for Realised Volatility and +0.044 for Composite. Holdings beta is contemporaneous and realised beta is trailing, so the gap remains a measurement diagnostic rather than an optimisation target.

## 9. Implementation and capacity

Capacity uses the frozen 1% participation assumption and lagged 21-day median dollar volume. Liquidity coverage is reported on trade days; no-trade days do not inflate the summary.

In [27]:
if MONITORING_SPECIFICATION.capacity_participation_rate != 0.01:
    raise ValueError("The frozen capacity participation assumption changed.")
common_implementation_state, common_liquidity_coverage = (
    calculate_implementation_monitoring_state(
        common_security_daily,
        factor_panel,
        portfolios=ACTIVE_PORTFOLIO_ORDER,
    )
)
capacity_column = "bottleneck_capacity_1pct_usd"
if capacity_column not in common_implementation_state:
    raise KeyError(f"Missing expected capacity column: {capacity_column}")

implementation_reconciliation = (
    common_baseline_daily[
        ["portfolio", "date", "turnover", "transaction_cost", "missing_return_weight"]
    ]
    .merge(
        common_implementation_state[
            ["portfolio", "date", "turnover", "transaction_cost", "missing_return_weight"]
        ],
        on=["portfolio", "date"],
        suffixes=("_portfolio", "_security"),
        validate="one_to_one",
    )
)
implementation_audit_rows = []
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    observed = implementation_reconciliation.loc[
        implementation_reconciliation["portfolio"].eq(portfolio)
    ]
    maximum_difference = max(
        (observed[f"{metric}_portfolio"] - observed[f"{metric}_security"]).abs().max()
        for metric in ("turnover", "transaction_cost", "missing_return_weight")
    )
    implementation_audit_rows.append(
        {
            "portfolio": portfolio,
            "observations": len(observed),
            "maximum_absolute_difference": maximum_difference,
            "audit_passes": bool(
                len(observed) == len(common_dates)
                and maximum_difference < AUDIT_TOLERANCE
            ),
        }
    )
implementation_audit = pd.DataFrame(implementation_audit_rows)
if not implementation_audit["audit_passes"].all():
    raise ValueError("The implementation diagnostics do not reconcile.")

display(implementation_audit)

,portfolio,observations,maximum_absolute_difference,audit_passes
0,Momentum Only,2635,2.220446e-16,True
1,Realised Volatility Only,2635,2.220446e-16,True
2,Composite Score,2635,2.220446e-16,True
3,Fixed 50/50 Sleeves,2635,2.220446e-16,True
4,Pure Inverse Volatility,2635,2.220446e-16,True


In [28]:
baseline_lookup = common_window_baseline.set_index("portfolio")
implementation_summary_rows = []
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_baseline_daily.loc[
        common_baseline_daily["portfolio"].eq(portfolio)
    ]
    state = common_implementation_state.loc[
        common_implementation_state["portfolio"].eq(portfolio)
    ]
    coverage = common_liquidity_coverage.loc[
        common_liquidity_coverage["portfolio"].eq(portfolio)
    ]
    trade_day_coverage = coverage.loc[coverage["turnover"].gt(AUDIT_TOLERANCE)]
    fully_covered_dates = trade_day_coverage.loc[
        trade_day_coverage["liquidity_coverage"].ge(1.0 - AUDIT_TOLERANCE),
        "date",
    ]
    capacity_observations = state.loc[
        state["date"].isin(fully_covered_dates), capacity_column
    ]
    implementation_summary_rows.append(
        {
            "portfolio": portfolio,
            "average_daily_turnover": daily["turnover"].mean(),
            "average_rebalance_turnover": daily.loc[
                daily["is_rebalance"], "turnover"
            ].mean(),
            "largest_trade_weight": state["largest_trade_weight"].max(),
            "annualised_turnover": daily["turnover"].mean() * TRADING_DAYS_PER_YEAR,
            "annualised_transaction_cost": (
                daily["transaction_cost"].mean() * TRADING_DAYS_PER_YEAR
            ),
            "annualised_return_cost_drag": baseline_lookup.loc[
                portfolio, "annualised_return_cost_drag"
            ],
            "minimum_trade_capacity_1pct_usd": capacity_observations.min(),
            "turnover_weighted_liquidity_coverage": (
                coverage["liquidity_covered_turnover"].sum()
                / coverage["turnover"].sum()
            ),
            "minimum_trade_day_liquidity_coverage": (
                trade_day_coverage["liquidity_coverage"].min()
            ),
            "maximum_missing_return_exposure": daily["missing_return_weight"].max(),
            "average_gross_exposure": daily["gross_exposure"].mean(),
        }
    )

implementation_capacity_summary = pd.DataFrame(implementation_summary_rows)
display(implementation_capacity_summary.round(6))

,portfolio,average_daily_turnover,average_rebalance_turnover,largest_trade_weight,annualised_turnover,annualised_transaction_cost,annualised_return_cost_drag,minimum_trade_capacity_1pct_usd,turnover_weighted_liquidity_coverage,minimum_trade_day_liquidity_coverage,maximum_missing_return_exposure,average_gross_exposure
0,Momentum Only,0.111424,0.557119,0.068628,28.078778,0.028079,0.028744,5.321390e+06,1.0,1.0,0.0,2.002969
1,Realised Volatility Only,0.082102,0.410509,0.136847,20.689643,0.020690,0.024269,1.590265e+07,1.0,1.0,0.0,2.000736
2,Composite Score,0.104652,0.523261,0.097766,26.372329,0.026372,0.030386,6.404006e+06,1.0,1.0,0.0,2.000719
3,Fixed 50/50 Sleeves,0.087102,0.435508,0.067011,21.949591,0.021950,0.024487,1.006031e+07,1.0,1.0,0.0,1.541543
4,Pure Inverse Volatility,0.092385,0.461923,0.059544,23.280896,0.023281,0.026005,9.424561e+06,1.0,1.0,0.0,1.607862


### Matched pairwise differences

Every value is the first named portfolio minus the second under the same dates, five-day phase, and 10 bps cost. For drawdown, a positive difference is shallower; for capacity, a positive difference is greater. No weighted score is applied.

In [29]:
comparison_metrics = (
    common_window_baseline.loc[
        common_window_baseline["portfolio"].isin(ACTIVE_PORTFOLIO_ORDER),
        [
            "portfolio",
            "annualised_return",
            "annualised_volatility",
            "sharpe_ratio",
            "max_drawdown",
        ],
    ]
    .merge(
        risk_concentration_summary[
            [
                "portfolio",
                "average_holdings_market_beta",
                "average_effective_position_count",
                "average_largest_absolute_sector_net_exposure",
                "average_top_five_beta_contribution_share",
                "average_top_five_return_contribution_share_63",
            ]
        ],
        on="portfolio",
        validate="one_to_one",
    )
    .merge(
        implementation_capacity_summary[
            [
                "portfolio",
                "average_daily_turnover",
                "minimum_trade_capacity_1pct_usd",
            ]
        ],
        on="portfolio",
        validate="one_to_one",
    )
    .set_index("portfolio")
)
pairwise_specifications = (
    ("Composite Score", "Realised Volatility Only"),
    ("Composite Score", "Momentum Only"),
    ("Fixed 50/50 Sleeves", "Realised Volatility Only"),
    ("Pure Inverse Volatility", "Realised Volatility Only"),
)
pairwise_metric_columns = {
    "net_annualised_return": "annualised_return",
    "annualised_volatility": "annualised_volatility",
    "sharpe_ratio": "sharpe_ratio",
    "max_drawdown": "max_drawdown",
    "average_daily_turnover": "average_daily_turnover",
    "holdings_market_beta": "average_holdings_market_beta",
    "effective_position_count": "average_effective_position_count",
    "largest_absolute_sector_net_exposure": (
        "average_largest_absolute_sector_net_exposure"
    ),
    "top_five_beta_contribution_share": (
        "average_top_five_beta_contribution_share"
    ),
    "top_five_return_contribution_share_63": (
        "average_top_five_return_contribution_share_63"
    ),
    "minimum_trade_capacity_1pct_usd": "minimum_trade_capacity_1pct_usd",
}
pairwise_difference_columns = {}
for first, second in pairwise_specifications:
    label = f"{first} minus {second}"
    pairwise_difference_columns[label] = {
        metric: (
            comparison_metrics.loc[first, column]
            - comparison_metrics.loc[second, column]
        )
        for metric, column in pairwise_metric_columns.items()
    }
pairwise_differences = (
    pd.DataFrame(pairwise_difference_columns)
    .rename_axis("metric")
    .reset_index()
)
if pairwise_differences.isna().any().any():
    raise ValueError("The pairwise difference table contains missing values.")

display(pairwise_differences.round(6))

,metric,Composite Score minus Realised Volatility Only,Composite Score minus Momentum Only,Fixed 50/50 Sleeves minus Realised Volatility Only,Pure Inverse Volatility minus Realised Volatility Only
0,net_annualised_return,-2.346800e-02,1.278820e-01,-5.730500e-02,-5.672300e-02
1,annualised_volatility,-3.531000e-02,-8.558000e-03,-8.021600e-02,-8.574600e-02
2,sharpe_ratio,-1.368500e-02,5.561000e-01,-5.472100e-02,-3.440300e-02
3,max_drawdown,1.522500e-01,2.032730e-01,2.019650e-01,2.628750e-01
4,average_daily_turnover,2.255000e-02,-6.772000e-03,5.000000e-03,1.028300e-02
5,holdings_market_beta,-1.583190e-01,5.432240e-01,-3.504490e-01,-3.942030e-01
6,effective_position_count,2.747000e-03,2.225000e-03,7.246493e+00,9.917555e+00
7,largest_absolute_sector_net_exposure,-3.215400e-02,6.674400e-02,-1.291650e-01,-1.389030e-01
8,top_five_beta_contribution_share,-3.948000e-03,1.221700e-02,2.897000e-02,1.303900e-02
9,top_five_return_contribution_share_63,-9.295000e-03,1.723400e-02,9.310000e-03,-4.479000e-03


### Implementation, capacity, and pairwise findings

- **Realised Volatility has the strongest implementation profile in this controlled baseline.** Its average daily and rebalance turnover are 8.21% and 41.05%, its annualised return cost drag is 2.43%, and its minimum fully covered 1% capacity is $15.90 million. Composite is costlier at 10.47%, 52.33%, 3.04%, and $6.40 million.
- **Liquidity and missing returns do not explain the differences.** Every strategy has 100% turnover-weighted and minimum trade-day liquidity coverage, with zero missing-return exposure. Security attribution and implementation totals reconcile below numerical tolerance.
- **Sleeve netting lowers realised gross exposure, not turnover relative to Realised Volatility.** Fixed and Pure average 1.54 and 1.61 gross, versus about 2.00 for the standalone and Composite portfolios, but their daily turnover is 8.71% and 9.24% and minimum capacity is $10.06 million and $9.42 million.
- **Composite versus Realised Volatility is a direct return-risk trade-off.** Composite trails by 2.35 percentage points of net annualised return and 0.014 Sharpe, but lowers volatility by 3.53 points, improves drawdown by 15.23 points, and reduces holdings beta by 0.158. Its capacity is $9.50 million lower.
- **The sleeves buy substantially greater protection at a larger return cost.** Fixed and Pure trail Realised Volatility by 5.73 and 5.67 return points and 0.055 and 0.034 Sharpe, while cutting volatility by 8.02 and 8.57 points, improving drawdown by 20.20 and 26.29 points, and lowering beta by 0.350 and 0.394.

Realised Volatility's advantage is therefore not an artefact of higher turnover, weak liquidity, missing returns, or lower costs applied by assumption. It is paired with higher beta, stronger sector tilts, and worse tail risk, but also with genuinely favourable capacity and cost characteristics.